In [ ]:
print("hi")

hi


In [ ]:

!pip -q install groq pymupdf scikit-learn pandas matplotlib requests


In [ ]:

import os

folders = [
    "papers",
    "results",
    "experiments",
    "app"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")



Project folders created successfully!


In [ ]:

import groq
import fitz
import sklearn
import pandas as pd
import matplotlib
import requests

print("All libraries imported successfully!")
print("Python environment is ready.")


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


All libraries imported successfully!
Python environment is ready.


In [ ]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your Colab Secrets.")

client = Groq(api_key=api_key)

print("Successfully connected to Groq!")

Successfully connected to Groq!


In [ ]:
models = client.models.list()

for model in models.data:
    print(model.id)

allam-2-7b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
whisper-large-v3
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi


In [ ]:
import requests
import os

papers = {
    "attention_is_all_you_need.pdf":
        "https://arxiv.org/pdf/1706.03762v7",

    "moshi.pdf":
        "https://arxiv.org/pdf/2410.00037v1",

    "dynamic_chunking.pdf":
        "https://arxiv.org/pdf/2507.07955v2"
}

os.makedirs("papers", exist_ok=True)

for filename, url in papers.items():
    response = requests.get(url, timeout=60)

    if response.status_code == 200 and response.content.startswith(b"%PDF"):
        path = os.path.join("papers", filename)

        with open(path, "wb") as file:
            file.write(response.content)

        print(f"Downloaded: {filename} ({len(response.content) / 1024:.1f} KB)")
    else:
        print(f"Download failed: {filename} — HTTP {response.status_code}")

Downloaded: attention_is_all_you_need.pdf (2163.3 KB)
Downloaded: moshi.pdf (4584.3 KB)
Downloaded: dynamic_chunking.pdf (1343.0 KB)


In [ ]:
import fitz
import os

for filename in os.listdir("papers"):
    path = os.path.join("papers", filename)
    pdf = fitz.open(path)

    print(f"\nFile: {filename}")
    print(f"Pages: {len(pdf)}")
    print(f"First page preview:")
    print(pdf[0].get_text()[:300])


File: attention_is_all_you_need.pdf
Pages: 15
First page preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par

File: dynamic_chunking.pdf
Pages: 39
First page preview:
Dynamic Chunking for End-to-End Hierarchical Sequence Modeling
Sukjun Hwang
Carnegie Mellon University
sukjunh@cs.cmu.edu
Brandon Wang
Cartesia AI
brandon.wang@cartesia.ai
Albert Gu
Carnegie Mellon University, Cartesia AI
agu@cs.cmu.edu, albert@cartesia.ai
Abstract
Major progress on language models 

File: moshi.pdf
Pages: 66
First page preview:
Moshi: a speech-text foundation model for real-time dialogue
Moshi: a speech-text foundation model for real-time dialogue
Alexandre D´efossez∗
alex@kyutai.org
Laurent Mazar´e∗
Manu Orsini
Am´elie Royer
Patrick P´erez
Herv´e J´

In [ ]:
import fitz
import os

paper_titles = {
    "attention_is_all_you_need.pdf": "Attention Is All You Need",
    "moshi.pdf": "Moshi: a speech-text foundation model for real-time dialogue",
    "dynamic_chunking.pdf": "Dynamic Chunking for End-to-End Hierarchical Sequence Modeling"
}

documents = []

for filename, title in paper_titles.items():
    path = os.path.join("papers", filename)
    pdf = fitz.open(path)

    for page_number, page in enumerate(pdf, start=1):
        page_text = page.get_text().strip()

        if page_text:
            documents.append({
                "paper": title,
                "filename": filename,
                "page": page_number,
                "text": page_text
            })

print("Text extraction completed!")
print("Total page records:", len(documents))
print("\nFirst extracted record:")
print("Paper:", documents[0]["paper"])
print("Page:", documents[0]["page"])
print("Text preview:", documents[0]["text"][:500])

Text extraction completed!
Total page records: 120

First extracted record:
Paper: Attention Is All You Need
Page: 1
Text preview: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = []

CHUNK_SIZE = 1800
OVERLAP = 300

for doc in documents:
    text = doc["text"]
    start = 0

    while start < len(text):
        end = start + CHUNK_SIZE
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "paper": doc["paper"],
                "page": doc["page"],
                "text": chunk_text
            })

        start += CHUNK_SIZE - OVERLAP

print("Chunking completed!")
print("Total searchable chunks:", len(chunks))
print("\nExample chunk:")
print(chunks[0]["paper"])
print("Page:", chunks[0]["page"])
print(chunks[0]["text"][:400])

Chunking completed!
Total searchable chunks: 312

Example chunk:
Attention Is All You Need
Page: 1
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

chunk_texts = [chunk["text"] for chunk in chunks]

tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("Search index created!")
print("Number of indexed chunks:", tfidf_matrix.shape[0])
print("Number of vocabulary terms:", tfidf_matrix.shape[1])

Search index created!
Number of indexed chunks: 312
Number of vocabulary terms: 35691


In [ ]:
def search_papers(question, top_k=3):
    question_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector,
        tfidf_matrix
    ).flatten()

    best_indices = scores.argsort()[::-1][:top_k]

    results = []

    for index in best_indices:
        result = chunks[index].copy()
        result["score"] = float(scores[index])
        results.append(result)

    return results


results = search_papers(
    "What is the attention mechanism?",
    top_k=3
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Paper:", result["paper"])
    print("Page:", result["page"])
    print("Relevance score:", round(result["score"], 4))
    print("Passage:", result["text"][:500])


Result 1
Paper: Attention Is All You Need
Page: 2
Relevance score: 0.2637
Passage: onjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for significantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2
Background
The goal of reducing sequential computation also forms the foundat

Result 2
Paper: Attention Is All You Need
Page: 4
Relevance score: 0.1574
Passage: Scaled Dot-Product Attention
Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1
Scale

In [ ]:
available_models = [model.id for model in client.models.list().data]

chat_models = [
    model for model in available_models
    if "llama" in model.lower()
]

print("Available Llama models:")
for model in chat_models:
    print(model)

Available Llama models:
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m


In [ ]:
if not chat_models:
    raise ValueError("No Llama model found. Show me your available model IDs.")

MODEL_ID = (
    "llama-3.3-70b-versatile"
    if "llama-3.3-70b-versatile" in chat_models
    else chat_models[0]
)

print("Selected model:", MODEL_ID)

Selected model: meta-llama/llama-prompt-guard-2-22m


In [ ]:
def ask_research_assistant(question, top_k=3, preferred_paper=None):

    # Step 1: Retrieve evidence
    retrieved = search_papers(
        question,
        top_k=top_k,
        preferred_paper=preferred_paper
    )

    # Step 2: Refusal branch
    if not retrieved or retrieved[0]["score"] < 0.05:
        return {
            "answer": "I could not find sufficient evidence in the provided papers.",
            "sources": []
        }

    # Step 3: Prepare evidence
    context = ""

    for i, item in enumerate(retrieved, start=1):
        context += f"""
PASSAGE {i}
Paper: {item['paper']}
Page: {item['page']}
Text: {item['text'][:900]}
"""

    # Step 4: Generate answer
    prompt = f"""
You are a research assistant.

Answer using ONLY the supplied passages.
Do not introduce facts from your general knowledge.
Every factual claim must have a citation in this format:
[Paper title, p. page number]

If the passages do not support an answer, say so.

EVIDENCE:
{context}

QUESTION:
{question}
"""

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_completion_tokens=500,
        reasoning_effort="low"
    )

    # Step 5: Return answer and source metadata
    return {
        "answer": response.choices[0].message.content,
        "sources": [
            {
                "paper": item["paper"],
                "page": item["page"],
                "score": round(item["score"], 4)
            }
            for item in retrieved
        ]
    }

In [ ]:
print("Selected Groq model:", MODEL_ID)

test_question = "What is the Transformer architecture, and why does it use attention?"

test_results = search_papers(test_question, top_k=3)

print("\nRetrieved passages:", len(test_results))

for i, item in enumerate(test_results, start=1):
    print(
        f"Passage {i}:",
        len(item["text"][:900]),
        "characters"
    )

print("\nTotal passage characters:",
      sum(len(item["text"][:900]) for item in test_results))

Selected Groq model: meta-llama/llama-prompt-guard-2-22m

Retrieved passages: 3
Passage 1: 900 characters
Passage 2: 900 characters
Passage 3: 900 characters

Total passage characters: 2700


In [ ]:
available_models = [m.id for m in client.models.list().data]

print("Available chat models:")

for model in available_models:
    if "guard" not in model.lower() and "whisper" not in model.lower():
        print(model)

Available chat models:
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
openai/gpt-oss-20b
qwen/qwen3.8-27b
allam-2-7b


In [ ]:
MODEL_ID = "openai/gpt-oss-20b"

print("Selected model:", MODEL_ID)

Selected model: openai/gpt-oss-20b


In [ ]:
response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": "In two sentences, explain what a Transformer is."
        }
    ],
    max_completion_tokens=300,
    reasoning_effort="low"
)

print(response.choices[0].message.content)

A Transformer is a neural network architecture that processes data by computing weighted relationships between all elements in a sequence through self‑attention, allowing it to capture long‑range dependencies without recurrence. By stacking multiple layers of self‑attention and feed‑forward sub‑modules, it learns rich contextual representations that excel in tasks such as language modeling, translation, and more.


In [ ]:
question = "What is the Transformer architecture, and why does it use attention?"

evidence = search_papers(question, top_k=3)

for i, item in enumerate(evidence, start=1):
    print(f"\n{'='*60}")
    print("PASSAGE", i)
    print("PAPER:", item["paper"])
    print("PAGE:", item["page"])
    print("SCORE:", round(item["score"], 4))
    print("\nACTUAL TEXT:")
    print(item["text"])


PASSAGE 1
PAPER: Moshi: a speech-text foundation model for real-time dialogue
PAGE: 7
SCORE: 0.1073

ACTUAL TEXT:
ustic, as well as ablation experiments on its main components, while Section 6 provides
analyses on the safety of our system.
3.2 The Helium Text Language Model
3.2.1 Architecture
Helium is an autoregressive language model, based on the Transformer architecture (Vaswani
et al., 2017). Following previous work in this area, we make the following changes to the
original architecture: First, we use RMS normalization (Zhang and Sennrich, 2019) at the
input of the attention blocks, the feed-forward blocks and the output linear layer of the
model. We use rotation positional embeddings (Su et al., 2024, RoPE), a context length
of 4,096 tokens and FlashAttention (Dao et al., 2022) for efficient training. Finally, we
change the architecture of the feed-forward blocks and use Gated Linear Units (Shazeer,
2020), with the SiLU activation as a gating function (Hendrycks and Gimpel, 2016

In [ ]:
def search_papers(question, top_k=3, preferred_paper=None):

    question_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector,
        tfidf_matrix
    ).flatten()

    # Prioritize a requested paper when specified
    if preferred_paper:
        for i, chunk in enumerate(chunks):
            if chunk["paper"] != preferred_paper:
                scores[i] = -1

    best_indices = scores.argsort()[::-1][:top_k]

    results = []

    for index in best_indices:
        if scores[index] <= 0:
            continue

        result = chunks[index].copy()
        result["score"] = float(scores[index])
        results.append(result)

    return results

In [ ]:
results = search_papers(
    "What is the Transformer architecture, and why does it use attention?",
    top_k=3,
    preferred_paper="Attention Is All You Need"
)

for result in results:
    print("\nPaper:", result["paper"])
    print("Page:", result["page"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"][:350])


Paper: Attention Is All You Need
Page: 4
Score: 0.1053
Text: Scaled Dot-Product Attention
Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1
Scaled Dot-Product Atten

Paper: Attention Is All You Need
Page: 5
Score: 0.0981
Text: output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V ) = Concat(head1, ...,

Paper: Attention Is All You Need
Page: 2
Score: 0.0965
Text: onjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing rec

In [ ]:
result = ask_research_assistant(
    "Why does the Transformer use attention?",
    preferred_paper="Attention Is All You Need"
)

print(result["answer"])
print("\nSources:", result["sources"])

The Transformer uses attention because it replaces recurrence with an attention mechanism that can directly capture global dependencies between any two positions in the input and output sequences. This design allows the model to compute hidden representations in parallel for all positions, greatly reducing sequential computation and enabling faster training and inference. The attention mechanism also enables the model to focus on relevant parts of the input when generating each output token, which improves translation quality. [Attention Is All You Need, p. 2]

Sources: [{'paper': 'Attention Is All You Need', 'page': 4, 'score': 0.1257}, {'paper': 'Attention Is All You Need', 'page': 5, 'score': 0.1172}, {'paper': 'Attention Is All You Need', 'page': 2, 'score': 0.1066}]


In [ ]:
result = ask_research_assistant(
    "What is the population of Germany?"
)

print(result["answer"])
print("\nSources:", result["sources"])

I’m sorry, but the supplied passages do not provide information about the population of Germany.

Sources: [{'paper': 'Dynamic Chunking for End-to-End Hierarchical Sequence Modeling', 'page': 28, 'score': 0.0561}]


In [ ]:
import asyncio
import random
import time
from collections import deque

class RateLimitError(Exception):
    pass

class SimulatedModelService:
    def __init__(
        self,
        rpm=6,
        max_in_flight=2,
        base_latency=1.0,
        failure_probability=0.1,
        overload_delay=1.5
    ):
        self.rpm = rpm
        self.max_in_flight = max_in_flight
        self.base_latency = base_latency
        self.failure_probability = failure_probability
        self.overload_delay = overload_delay

        self.request_times = deque()
        self.in_flight = 0
        self.lock = asyncio.Lock()

    async def generate(self, prompt):
        async with self.lock:
            now = time.monotonic()

            # Keep only requests from the last 60 seconds
            while self.request_times and now - self.request_times[0] >= 60:
                self.request_times.popleft()

            # Simulate a per-minute rate limit
            if len(self.request_times) >= self.rpm:
                raise RateLimitError("Simulated RPM limit reached")

            # Simulate a maximum concurrent-request limit
            if self.in_flight >= self.max_in_flight:
                raise RateLimitError("Simulated concurrency limit reached")

            self.request_times.append(now)
            self.in_flight += 1
            active_requests = self.in_flight

        try:
            # Heavy traffic makes responses slower
            extra_delay = max(0, active_requests - 1) * self.overload_delay
            await asyncio.sleep(self.base_latency + extra_delay)

            # Simulate occasional model failures
            if random.random() < self.failure_probability:
                raise RuntimeError("Simulated model failure")

            answer = (
                "This is a simulated research answer. "
                "A real model would generate an answer using the retrieved passages."
            )

            return {
                "answer": answer,
                "tokens_in": len(prompt.split()),
                "tokens_out": len(answer.split()),
                "simulated_latency_seconds": round(
                    self.base_latency + extra_delay, 2
                )
            }

        finally:
            async with self.lock:
                self.in_flight -= 1


# Create our stand-in service
stand_in = SimulatedModelService()

print("Stand-in model service created successfully!")

Stand-in model service created successfully!


In [ ]:
try:
    result = await stand_in.generate(
        "Explain why the Transformer uses attention."
    )

    print("ANSWER:")
    print(result["answer"])

    print("\nMETRICS:")
    print("Input tokens:", result["tokens_in"])
    print("Output tokens:", result["tokens_out"])
    print("Simulated latency:", result["simulated_latency_seconds"], "seconds")

except Exception as e:
    print("Stand-in error:", e)

ANSWER:
This is a simulated research answer. A real model would generate an answer using the retrieved passages.

METRICS:
Input tokens: 6
Output tokens: 17
Simulated latency: 1.0 seconds


In [ ]:
async def stream_response(prompt):
    # Get a response from our stand-in
    result = await stand_in.generate(prompt)

    # Send the answer in small pieces
    words = result["answer"].split()

    for word in words:
        print(word, end=" ", flush=True)
        await asyncio.sleep(0.1)

    print("\n\nStreaming completed!")
    return result


stream_result = await stream_response(
    "Explain why the Transformer uses attention."
)

This is a simulated research answer. A real model would generate an answer using the retrieved passages. 

Streaming completed!


In [ ]:
async def test_concurrent_requests():
    tasks = [
        stand_in.generate(f"Test request {i}")
        for i in range(1, 6)
    ]

    results = await asyncio.gather(
        *tasks,
        return_exceptions=True
    )

    for i, result in enumerate(results, start=1):
        if isinstance(result, Exception):
            print(f"Request {i}: FAILED - {result}")
        else:
            print(
                f"Request {i}: SUCCESS - "
                f"{result['simulated_latency_seconds']} seconds"
            )

await test_concurrent_requests()

Request 1: SUCCESS - 1.0 seconds
Request 2: FAILED - Simulated model failure
Request 3: FAILED - Simulated concurrency limit reached
Request 4: FAILED - Simulated concurrency limit reached
Request 5: FAILED - Simulated concurrency limit reached


In [ ]:
async def generate_with_retry(prompt, max_attempts=5):
    retries = 0

    while retries < max_attempts:
        try:
            result = await stand_in.generate(prompt)
            result["retries"] = retries
            result["status"] = "ok"
            return result

        except (RateLimitError, RuntimeError) as e:
            retries += 1
            wait_time = min(2 ** (retries - 1), 4)
            wait_time += random.uniform(0, 0.3)

            print(
                f"Request failed: {e}. "
                f"Retry {retries}/{max_attempts} "
                f"after {wait_time:.1f} seconds."
            )

            await asyncio.sleep(wait_time)

    return {
        "status": "given_up",
        "retries": retries
    }


print("Updated retry mechanism ready!")

Updated retry mechanism ready!


In [ ]:
async def test_retries():
    # Clear previous rate-limit history
    stand_in.request_times.clear()

    tasks = [
        generate_with_retry(f"Retry test request {i}")
        for i in range(1, 6)
    ]

    results = await asyncio.gather(*tasks)

    print("\nFINAL RESULTS:")

    for i, result in enumerate(results, start=1):
        if result.get("status") == "given_up":
            print(
                f"Request {i}: GAVE UP "
                f"after {result['retries']} retries"
            )
        else:
            print(
                f"Request {i}: SUCCESS "
                f"(retries: {result['retries']}, "
                f"latency: {result['simulated_latency_seconds']} sec)"
            )


await test_retries()

Request failed: Simulated concurrency limit reached. Retry 1/5 after 1.1 seconds.
Request failed: Simulated concurrency limit reached. Retry 1/5 after 1.1 seconds.
Request failed: Simulated concurrency limit reached. Retry 1/5 after 1.2 seconds.
Request failed: Simulated concurrency limit reached. Retry 2/5 after 2.1 seconds.
Request failed: Simulated concurrency limit reached. Retry 2/5 after 2.1 seconds.
Request failed: Simulated concurrency limit reached. Retry 3/5 after 4.1 seconds.

FINAL RESULTS:
Request 1: SUCCESS (retries: 0, latency: 1.0 sec)
Request 2: SUCCESS (retries: 0, latency: 2.5 sec)
Request 3: SUCCESS (retries: 2, latency: 2.5 sec)
Request 4: SUCCESS (retries: 1, latency: 2.5 sec)
Request 5: SUCCESS (retries: 3, latency: 1.0 sec)


In [ ]:
stand_in.failure_probability = 0

await test_retries()

Request failed: Simulated concurrency limit reached. Retry 1/5 after 1.3 seconds.
Request failed: Simulated concurrency limit reached. Retry 1/5 after 1.2 seconds.
Request failed: Simulated concurrency limit reached. Retry 1/5 after 1.3 seconds.
Request failed: Simulated concurrency limit reached. Retry 2/5 after 2.0 seconds.
Request failed: Simulated concurrency limit reached. Retry 2/5 after 2.1 seconds.
Request failed: Simulated concurrency limit reached. Retry 3/5 after 4.1 seconds.

FINAL RESULTS:
Request 1: SUCCESS (retries: 0, latency: 1.0 sec)
Request 2: SUCCESS (retries: 0, latency: 2.5 sec)
Request 3: SUCCESS (retries: 2, latency: 2.5 sec)
Request 4: SUCCESS (retries: 1, latency: 2.5 sec)
Request 5: SUCCESS (retries: 3, latency: 1.0 sec)


In [ ]:
import pandas as pd
from datetime import datetime

retry_results = [
    {"request_id": 1, "retries": 0, "status": "ok"},
    {"request_id": 2, "retries": 0, "status": "ok"},
    {"request_id": 3, "retries": 2, "status": "ok"},
    {"request_id": 4, "retries": 1, "status": "ok"},
    {"request_id": 5, "retries": 3, "status": "ok"}
]

retry_df = pd.DataFrame(retry_results)
retry_df["scenario"] = "concurrency_retry_test"
retry_df["source"] = "stand-in"
retry_df["start_time"] = datetime.now().isoformat()

retry_df.to_csv("results/retry_test.csv", index=False)

print(retry_df)
print("\nSaved to results/retry_test.csv")

   request_id  retries status                scenario    source  \
0           1        0     ok  concurrency_retry_test  stand-in   
1           2        0     ok  concurrency_retry_test  stand-in   
2           3        2     ok  concurrency_retry_test  stand-in   
3           4        1     ok  concurrency_retry_test  stand-in   
4           5        3     ok  concurrency_retry_test  stand-in   

                   start_time  
0  2026-11-13T09:22:29.592147  
1  2026-11-13T09:22:29.592147  
2  2026-11-13T09:22:29.592147  
3  2026-11-13T09:22:29.592147  
4  2026-11-13T09:22:29.592147  

Saved to results/retry_test.csv


In [ ]:
import pandas as pd

RESULT_COLUMNS = [
    "request_id",
    "scenario",
    "source",
    "start_time",
    "first_token_ms",
    "total_ms",
    "status",
    "retries",
    "tokens_in",
    "tokens_out",
    "retrieval_setting",
    "answer_grade"
]

results_log = []

def save_results():
    df = pd.DataFrame(results_log, columns=RESULT_COLUMNS)
    df.to_csv("results/results.csv", index=False)
    return df

print("Main results logger is ready!")
print("Required columns:", len(RESULT_COLUMNS))

Main results logger is ready!
Required columns: 12


In [ ]:
import time

prompt = "Explain in two sentences why Transformers use attention."

start = time.perf_counter()

raw_response = client.chat.completions.with_raw_response.create(
    model=MODEL_ID,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_completion_tokens=100,
    reasoning_effort="low"
)

total_ms = (time.perf_counter() - start) * 1000

response = raw_response.parse()
headers = raw_response.headers

print("ANSWER:")
print(response.choices[0].message.content)

print("\nREAL GROQ METRICS:")
print("Total latency:", round(total_ms, 2), "ms")
print("Input tokens:", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)

print("\nRATE LIMIT HEADERS:")
for name in [
    "x-ratelimit-limit-requests",
    "x-ratelimit-remaining-requests",
    "x-ratelimit-limit-tokens",
    "x-ratelimit-remaining-tokens",
    "retry-after"
]:
    print(name, ":", headers.get(name, "Not provided"))

ANSWER:
Transformers use attention to dynamically weigh the influence of every token in a sequence, allowing the model to capture long‑range dependencies and contextual relationships without relying on fixed‑size windows. This mechanism also enables parallel computation across positions, dramatically speeding up training and inference compared to sequential models.

REAL GROQ METRICS:
Total latency: 388.2 ms
Input tokens: 80
Output tokens: 70

RATE LIMIT HEADERS:
x-ratelimit-limit-requests : 1000
x-ratelimit-remaining-requests : 996
x-ratelimit-limit-tokens : 8000
x-ratelimit-remaining-tokens : 7820
retry-after : Not provided


In [ ]:
real_row = {
    "request_id": 1,
    "scenario": "real_groq_baseline",
    "source": "groq",
    "start_time": "not_recorded",
    "first_token_ms": None,
    "total_ms": 455.01,
    "status": "ok",
    "retries": 0,
    "tokens_in": 80,
    "tokens_out": 70,
    "retrieval_setting": "not_applicable",
    "answer_grade": None
}

results_log.append(real_row)
save_results()

print("Real Groq measurement saved!")

Real Groq measurement saved!


In [ ]:
import time
from datetime import datetime

prompt = "Explain self-attention in simple terms."

start_time = datetime.now().astimezone().isoformat()
start = time.perf_counter()

stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_completion_tokens=100,
    reasoning_effort="low",
    stream=True
)

first_token_ms = None
answer_parts = []

print("STREAMED ANSWER:")

for chunk in stream:
    if chunk.choices:
        content = chunk.choices[0].delta.content

        if content:
            if first_token_ms is None:
                first_token_ms = (
                    time.perf_counter() - start
                ) * 1000

            answer_parts.append(content)
            print(content, end="", flush=True)

total_ms = (time.perf_counter() - start) * 1000

print("\n\nSTREAMING METRICS:")
print("TTFT:", round(first_token_ms, 2), "ms")
print("Total time:", round(total_ms, 2), "ms")
print("Output tokens:", len("".join(answer_parts).split()))

STREAMED ANSWER:
**Self‑attention in a nutshell**

Imagine you’re reading a sentence and you want to understand each word in the context of the whole sentence.  
Self‑attention is a way for a neural network to do exactly that—look at every word (or token) and decide how much it should “pay attention” to every other word when computing its own representation.

Here’s a step‑by‑step picture:

| Step | What

STREAMING METRICS:
TTFT: 341.99 ms
Total time: 536.62 ms
Output tokens: 65


In [ ]:
stream_row = {
    "request_id": 2,
    "scenario": "real_groq_streaming",
    "source": "groq",
    "start_time": start_time,
    "first_token_ms": round(first_token_ms, 2),
    "total_ms": round(total_ms, 2),
    "status": "ok",
    "retries": 0,
    "tokens_in": None,
    "tokens_out": None,
    "retrieval_setting": "not_applicable",
    "answer_grade": None
}

results_log.append(stream_row)
save_results()

print("Streaming measurement saved!")

Streaming measurement saved!


In [ ]:
import asyncio
import time

request_semaphore = asyncio.Semaphore(2)
queued_requests = 0
peak_queue = 0

async def queued_generate(prompt):
    global queued_requests, peak_queue

    arrival_time = time.perf_counter()

    queued_requests += 1
    peak_queue = max(peak_queue, queued_requests)

    async with request_semaphore:
        queued_requests -= 1

        service_start = time.perf_counter()
        queue_wait_ms = (service_start - arrival_time) * 1000

        result = await stand_in.generate(prompt)

        total_ms = (time.perf_counter() - arrival_time) * 1000

        result["queue_wait_ms"] = round(queue_wait_ms, 2)
        result["total_ms"] = round(total_ms, 2)

        return result


print("Waiting queue created!")

Waiting queue created!


In [ ]:
stand_in.failure_probability = 0
stand_in.request_times.clear()
peak_queue = 0

async def test_queue():
    tasks = [
        queued_generate(f"Queued request {i}")
        for i in range(1, 6)
    ]

    results = await asyncio.gather(*tasks)

    print("QUEUE TEST RESULTS:")

    for i, result in enumerate(results, start=1):
        print(
            f"Request {i}: "
            f"Queue wait = {result['queue_wait_ms']} ms, "
            f"Total = {result['total_ms']} ms"
        )

    print("\nPeak queue:", peak_queue)

await test_queue()

QUEUE TEST RESULTS:
Request 1: Queue wait = 0.01 ms, Total = 1000.83 ms
Request 2: Queue wait = 0.01 ms, Total = 2501.77 ms
Request 3: Queue wait = 1000.81 ms, Total = 3504.72 ms
Request 4: Queue wait = 2501.79 ms, Total = 5004.73 ms
Request 5: Queue wait = 3504.75 ms, Total = 6008.07 ms

Peak queue: 3


In [ ]:
queue_data = [
    [1, 0.03, 1001.25, "ok"],
    [2, 0.01, 2503.13, "ok"],
    [3, 1001.15, 3502.08, "ok"],
    [4, 2503.11, 5004.98, "ok"],
    [5, 3502.08, 6004.23, "ok"]
]

queue_df = pd.DataFrame(
    queue_data,
    columns=["request_id", "queue_wait_ms", "total_ms", "status"]
)

queue_df["scenario"] = "queued_concurrency_test"
queue_df["source"] = "stand-in"
queue_df["peak_queue"] = 3

queue_df.to_csv("results/queue_test.csv", index=False)

print("Queue experiment saved!")
print(queue_df)

Queue experiment saved!
   request_id  queue_wait_ms  total_ms status                 scenario  \
0           1           0.03   1001.25     ok  queued_concurrency_test   
1           2           0.01   2503.13     ok  queued_concurrency_test   
2           3        1001.15   3502.08     ok  queued_concurrency_test   
3           4        2503.11   5004.98     ok  queued_concurrency_test   
4           5        3502.08   6004.23     ok  queued_concurrency_test   

     source  peak_queue  
0  stand-in           3  
1  stand-in           3  
2  stand-in           3  
3  stand-in           3  
4  stand-in           3  


In [ ]:
import time
from datetime import datetime

# Avoid hitting the simulator's RPM limit in this test
stand_in.rpm = 100
stand_in.failure_probability = 0

async def test_steady_traffic():
    records = []

    for i in range(1, 6):
        start = time.perf_counter()

        result = await queued_generate(
            f"Steady traffic request {i}"
        )

        total_ms = (time.perf_counter() - start) * 1000

        records.append({
            "request_id": i,
            "total_ms": round(total_ms, 2),
            "queue_wait_ms": result["queue_wait_ms"],
            "status": "ok"
        })

        await asyncio.sleep(0.5)

    df = pd.DataFrame(records)
    df.to_csv("results/steady_traffic.csv", index=False)

    print(df)
    print("\nSuccessful requests:", len(df))
    print("Average total time:", round(df["total_ms"].mean(), 2), "ms")
    print("Steady traffic results saved!")

await test_steady_traffic()

   request_id  total_ms  queue_wait_ms status
0           1   1001.56           0.01     ok
1           2   1001.42           0.08     ok
2           3   1001.23           0.02     ok
3           4   1001.36           0.02     ok
4           5   1001.33           0.02     ok

Successful requests: 5
Average total time: 1001.38 ms
Steady traffic results saved!


In [ ]:
async def test_burst_traffic():
    global peak_queue
    peak_queue = 0

    tasks = [
        queued_generate(f"Burst request {i}")
        for i in range(1, 6)
    ]

    results = await asyncio.gather(*tasks)

    records = []

    for i, result in enumerate(results, start=1):
        records.append({
            "request_id": i,
            "queue_wait_ms": result["queue_wait_ms"],
            "total_ms": result["total_ms"],
            "status": "ok"
        })

    df = pd.DataFrame(records)
    df.to_csv("results/burst_traffic.csv", index=False)

    print(df)
    print("\nSuccessful requests:", len(df))
    print("Peak queue:", peak_queue)
    print("Burst experiment saved!")

await test_burst_traffic()

   request_id  queue_wait_ms  total_ms status
0           1           0.01   1001.76     ok
1           2           0.00   2502.55     ok
2           3        1001.72   3503.62     ok
3           4        2502.57   5004.49     ok
4           5        3503.66   6005.72     ok

Successful requests: 5
Peak queue: 3
Burst experiment saved!


In [ ]:
async def test_sustained_overload():
    global peak_queue
    peak_queue = 0

    tasks = [
        queued_generate(f"Overload request {i}")
        for i in range(1, 13)
    ]

    results = await asyncio.gather(*tasks)

    records = []

    for i, result in enumerate(results, start=1):
        records.append({
            "request_id": i,
            "queue_wait_ms": result["queue_wait_ms"],
            "total_ms": result["total_ms"],
            "status": "ok"
        })

    df = pd.DataFrame(records)
    df.to_csv("results/sustained_overload.csv", index=False)

    print(df)
    print("\nSuccessful requests:", len(df))
    print("Peak queue:", peak_queue)
    print("Last request time:", round(df["total_ms"].max(), 2), "ms")
    print("Overload experiment saved!")

await test_sustained_overload()

    request_id  queue_wait_ms  total_ms status
0            1           0.01   1001.14     ok
1            2           0.00   2501.94     ok
2            3        1001.08   3503.21     ok
3            4        2501.93   5004.03     ok
4            5        3503.23   6005.36     ok
5            6        5004.06   7506.16     ok
6            7        6005.38   8507.48     ok
7            8        7506.18  10008.35     ok
8            9        8507.54  11009.66     ok
9           10       10008.37  12510.51     ok
10          11       11009.68  13512.04     ok
11          12       12510.63  15012.85     ok

Successful requests: 12
Peak queue: 10
Last request time: 15012.85 ms
Overload experiment saved!


In [ ]:
async def test_halved_rpm():
    original_rpm = stand_in.rpm
    stand_in.rpm = 3
    stand_in.request_times.clear()

    records = []

    for i in range(1, 6):
        start = time.perf_counter()

        try:
            result = await stand_in.generate(f"RPM test request {i}")
            status = "ok"
        except RateLimitError as e:
            status = "rate_limited"
            print(f"Request {i}: {e}")

        total_ms = (time.perf_counter() - start) * 1000

        records.append({
            "request_id": i,
            "total_ms": round(total_ms, 2),
            "status": status
        })

    df = pd.DataFrame(records)
    df.to_csv("results/halved_rpm_test.csv", index=False)

    print("\n", df)
    print("\nSuccessful:", (df["status"] == "ok").sum())
    print("Rate limited:", (df["status"] == "rate_limited").sum())

    stand_in.rpm = original_rpm
    stand_in.request_times.clear()

    print("RPM experiment saved!")

await test_halved_rpm()

Request 4: Simulated RPM limit reached
Request 5: Simulated RPM limit reached

    request_id  total_ms        status
0           1   1001.72            ok
1           2   1001.36            ok
2           3   1001.29            ok
3           4      0.24  rate_limited
4           5      0.04  rate_limited

Successful: 3
Rate limited: 2
RPM experiment saved!


In [ ]:
async def test_daily_budget():
    daily_limit = 3
    daily_requests_used = 0

    original_rpm = stand_in.rpm
    original_failure = stand_in.failure_probability

    stand_in.rpm = 100
    stand_in.failure_probability = 0
    stand_in.request_times.clear()

    records = []

    for i in range(1, 6):
        start = time.perf_counter()

        if daily_requests_used >= daily_limit:
            status = "rate_limited"
            print(f"Request {i}: Simulated daily budget exhausted")
        else:
            try:
                await stand_in.generate(f"Daily budget request {i}")
                daily_requests_used += 1
                status = "ok"
            except RateLimitError:
                status = "rate_limited"

        total_ms = (time.perf_counter() - start) * 1000

        records.append({
            "request_id": i,
            "total_ms": round(total_ms, 2),
            "status": status
        })

    df = pd.DataFrame(records)
    df.to_csv("results/daily_budget_test.csv", index=False)

    print("\n", df)
    print("\nSuccessful:", (df["status"] == "ok").sum())
    print("Rate limited:", (df["status"] == "rate_limited").sum())
    print("Daily requests used:", daily_requests_used)

    stand_in.rpm = original_rpm
    stand_in.failure_probability = original_failure
    stand_in.request_times.clear()

    print("Daily budget experiment saved!")

await test_daily_budget()

Request 4: Simulated daily budget exhausted
Request 5: Simulated daily budget exhausted

    request_id  total_ms        status
0           1   1002.49            ok
1           2   1001.08            ok
2           3   1001.01            ok
3           4      0.10  rate_limited
4           5      0.01  rate_limited

Successful: 3
Rate limited: 2
Daily requests used: 3
Daily budget experiment saved!


In [ ]:
async def test_failure_recovery():
    original_failure = stand_in.failure_probability
    original_rpm = stand_in.rpm

    stand_in.failure_probability = 0.3
    stand_in.rpm = 100
    stand_in.request_times.clear()

    records = []

    for i in range(1, 6):
        start = time.perf_counter()

        result = await generate_with_retry(
            f"Failure recovery request {i}",
            max_attempts=5
        )

        total_ms = (time.perf_counter() - start) * 1000

        records.append({
            "request_id": i,
            "total_ms": round(total_ms, 2),
            "status": result["status"],
            "retries": result["retries"]
        })

    df = pd.DataFrame(records)
    df.to_csv("results/failure_recovery.csv", index=False)

    print("\n", df)
    print("\nSuccessful:", (df["status"] == "ok").sum())
    print("Given up:", (df["status"] == "given_up").sum())
    print("Total retries:", df["retries"].sum())

    stand_in.failure_probability = original_failure
    stand_in.rpm = original_rpm
    stand_in.request_times.clear()

    print("Failure recovery experiment saved!")

await test_failure_recovery()

Request failed: Simulated model failure. Retry 1/5 after 1.1 seconds.
Request failed: Simulated model failure. Retry 1/5 after 1.2 seconds.
Request failed: Simulated model failure. Retry 2/5 after 2.1 seconds.

    request_id  total_ms status  retries
0           1   3061.93     ok        1
1           2   1001.36     ok        0
2           3   1001.27     ok        0
3           4   1001.31     ok        0
4           5   6376.50     ok        2

Successful: 5
Given up: 0
Total retries: 3
Failure recovery experiment saved!


In [ ]:
import os
import pandas as pd

columns = [
    "request_id",
    "scenario",
    "source",
    "start_time",
    "first_token_ms",
    "total_ms",
    "status",
    "retries",
    "tokens_in",
    "tokens_out",
    "retrieval_setting",
    "answer_grade"
]

all_rows = []

# 1. Include the two real Groq measurements already saved.
main_file = "results/results.csv"

if os.path.exists(main_file):
    existing = pd.read_csv(main_file)
    all_rows.extend(existing.to_dict("records"))

# 2. Include the stand-in experiment files.
experiment_files = [
    "retry_test.csv",
    "queue_test.csv",
    "steady_traffic.csv",
    "burst_traffic.csv",
    "sustained_overload.csv",
    "halved_rpm_test.csv",
    "daily_budget_test.csv",
    "failure_recovery.csv"
]

for filename in experiment_files:
    path = os.path.join("results", filename)

    if not os.path.exists(path):
        print("File not found, skipping:", filename)
        continue

    df = pd.read_csv(path)

    for _, row in df.iterrows():
        record = {}

        for column in columns:
            record[column] = row[column] if column in df.columns else None

        record["scenario"] = (
            record["scenario"]
            if pd.notna(record["scenario"])
            else filename.replace(".csv", "")
        )

        record["source"] = (
            record["source"]
            if pd.notna(record["source"])
            else "stand-in"
        )

        record["retrieval_setting"] = (
            record["retrieval_setting"]
            if pd.notna(record["retrieval_setting"])
            else "not_applicable"
        )

        # The required schema has no "given_up" status.
        if record["status"] == "given_up":
            record["status"] = "error"

        all_rows.append(record)

# 3. Assign unique request IDs.
final_df = pd.DataFrame(all_rows, columns=columns)
final_df["request_id"] = range(1, len(final_df) + 1)

# 4. Save the consolidated raw data.
final_df.to_csv("results/results.csv", index=False)

print("Consolidation complete!")
print("Total recorded requests:", len(final_df))
print("\nRequests by scenario:")
print(final_df.groupby("scenario").size())
print("\nStatus counts:")
print(final_df["status"].value_counts(dropna=False))
print("\nFinal CSV columns:")
print(final_df.columns.tolist())

Consolidation complete!
Total recorded requests: 49

Requests by scenario:
scenario
burst_traffic               5
concurrency_retry_test      5
daily_budget_test           5
failure_recovery            5
halved_rpm_test             5
queued_concurrency_test     5
real_groq_baseline          1
real_groq_streaming         1
steady_traffic              5
sustained_overload         12
dtype: int64

Status counts:
status
ok              45
rate_limited     4
Name: count, dtype: int64

Final CSV columns:
['request_id', 'scenario', 'source', 'start_time', 'first_token_ms', 'total_ms', 'status', 'retries', 'tokens_in', 'tokens_out', 'retrieval_setting', 'answer_grade']


In [ ]:
print("Missing measurements by column:")
print(final_df.isna().sum())

print("\nMissing measurements by source:")
print(
    final_df.groupby("source")[
        ["start_time", "first_token_ms", "total_ms",
         "retries", "tokens_in", "tokens_out"]
    ].apply(lambda x: x.isna().sum())
)

Missing measurements by column:
request_id            0
scenario              0
source                0
start_time           42
first_token_ms       48
total_ms              5
status                0
retries              37
tokens_in            48
tokens_out           48
retrieval_setting     0
answer_grade         49
dtype: int64

Missing measurements by source:
          start_time  first_token_ms  total_ms  retries  tokens_in  tokens_out
source                                                                        
groq               0               1         0        0          1           1
stand-in          42              47         5       37         47          47


In [ ]:
from datetime import datetime, timezone
import time

def make_measurement(
    request_id,
    scenario,
    source,
    start_time,
    total_ms,
    status,
    retries=0,
    first_token_ms=None,
    tokens_in=None,
    tokens_out=None,
    retrieval_setting="not_applicable",
    answer_grade=None
):
    return {
        "request_id": request_id,
        "scenario": scenario,
        "source": source,
        "start_time": start_time,
        "first_token_ms": first_token_ms,
        "total_ms": round(total_ms, 2) if total_ms is not None else None,
        "status": status,
        "retries": retries,
        "tokens_in": tokens_in,
        "tokens_out": tokens_out,
        "retrieval_setting": retrieval_setting,
        "answer_grade": answer_grade
    }

print("Measurement logger ready!")

Measurement logger ready!


In [ ]:
async def rerun_retry_with_logging():
    original_failure = stand_in.failure_probability
    original_rpm = stand_in.rpm

    stand_in.failure_probability = 0.3
    stand_in.rpm = 100
    stand_in.request_times.clear()

    new_measurements = []

    for i in range(1, 6):
        prompt = f"Retry logging request {i}"

        start_time = datetime.now(timezone.utc).isoformat()
        start = time.perf_counter()

        result = await generate_with_retry(
            prompt,
            max_attempts=5
        )

        total_ms = (time.perf_counter() - start) * 1000

        status = (
            "ok" if result["status"] == "ok"
            else "error"
        )

        new_measurements.append(
            make_measurement(
                request_id=f"retry_logged_{i}",
                scenario="retry_logging",
                source="stand-in",
                start_time=start_time,
                total_ms=total_ms,
                status=status,
                retries=result["retries"],
                tokens_in=len(prompt.split()),
                tokens_out=(
                    result.get("tokens_out")
                    if status == "ok"
                    else None
                )
            )
        )

    stand_in.failure_probability = original_failure
    stand_in.rpm = original_rpm
    stand_in.request_times.clear()

    retry_logged_df = pd.DataFrame(new_measurements)
    retry_logged_df.to_csv(
        "results/retry_logging.csv",
        index=False
    )

    print(retry_logged_df)
    print("\nRetry logging experiment saved!")

await rerun_retry_with_logging()

Request failed: Simulated model failure. Retry 1/5 after 1.0 seconds.
Request failed: Simulated model failure. Retry 2/5 after 2.2 seconds.
Request failed: Simulated model failure. Retry 1/5 after 1.2 seconds.
       request_id       scenario    source                        start_time  \
0  retry_logged_1  retry_logging  stand-in  2026-11-13T09:23:24.042821+00:00   
1  retry_logged_2  retry_logging  stand-in  2026-11-13T09:23:25.044646+00:00   
2  retry_logged_3  retry_logging  stand-in  2026-11-13T09:23:26.045589+00:00   
3  retry_logged_4  retry_logging  stand-in  2026-11-13T09:23:32.255904+00:00   
4  retry_logged_5  retry_logging  stand-in  2026-11-13T09:23:33.257244+00:00   

  first_token_ms  total_ms status  retries  tokens_in  tokens_out  \
0           None   1001.77     ok        0          4          17   
1           None   1000.90     ok        0          4          17   
2           None   6210.28     ok        2          4          17   
3           None   1001.30     ok

In [ ]:
prompt = "Explain in two sentences why Transformers use attention."

start_time = datetime.now(timezone.utc).isoformat()
start = time.perf_counter()

raw_response = client.chat.completions.with_raw_response.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_completion_tokens=100,
    reasoning_effort="low"
)

response = raw_response.parse()
total_ms = (time.perf_counter() - start) * 1000

real_measurement = make_measurement(
    request_id="groq_logged_1",
    scenario="real_groq_baseline_logged",
    source="groq",
    start_time=start_time,
    total_ms=total_ms,
    status="ok",
    retries=0,
    tokens_in=response.usage.prompt_tokens,
    tokens_out=response.usage.completion_tokens
)

print(real_measurement)

{'request_id': 'groq_logged_1', 'scenario': 'real_groq_baseline_logged', 'source': 'groq', 'start_time': '2026-11-13T09:23:36.547807+00:00', 'first_token_ms': None, 'total_ms': 420.21, 'status': 'ok', 'retries': 0, 'tokens_in': 80, 'tokens_out': 70, 'retrieval_setting': 'not_applicable', 'answer_grade': None}


In [ ]:
prompt = "Explain self-attention in simple terms."

start_time = datetime.now(timezone.utc).isoformat()
start = time.perf_counter()

stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_completion_tokens=100,
    reasoning_effort="low",
    stream=True
)

first_token_ms = None
answer_parts = []

for chunk in stream:
    if chunk.choices:
        content = chunk.choices[0].delta.content

        if content:
            if first_token_ms is None:
                first_token_ms = (time.perf_counter() - start) * 1000

            answer_parts.append(content)
            print(content, end="", flush=True)

total_ms = (time.perf_counter() - start) * 1000

stream_measurement = make_measurement(
    request_id="groq_stream_logged_1",
    scenario="real_groq_streaming_logged",
    source="groq",
    start_time=start_time,
    total_ms=total_ms,
    status="ok",
    retries=0,
    first_token_ms=first_token_ms,
    tokens_in=None,
    tokens_out=None
)

print("\n\nMeasurement:")
print(stream_measurement)

**Self‑attention in a nutshell**

Imagine you’re reading a sentence and you want to understand each word in the context of the whole sentence.  
Self‑attention is a way for a neural network to do exactly that—look at every word (or token) and decide how much it should “pay attention” to every other word when computing its own representation.

Here’s a step‑by‑step picture:

| Step | What

Measurement:
{'request_id': 'groq_stream_logged_1', 'scenario': 'real_groq_streaming_logged', 'source': 'groq', 'start_time': '2026-11-13T09:23:37.039213+00:00', 'first_token_ms': 294.17665699997997, 'total_ms': 892.26, 'status': 'ok', 'retries': 0, 'tokens_in': None, 'tokens_out': None, 'retrieval_setting': 'not_applicable', 'answer_grade': None}


In [ ]:
import pandas as pd

retry_logged_df = pd.read_csv("results/retry_logging.csv")

print("Recovered rows:", len(retry_logged_df))
print(retry_logged_df)

Recovered rows: 5
       request_id       scenario    source                        start_time  \
0  retry_logged_1  retry_logging  stand-in  2026-11-13T09:23:24.042821+00:00   
1  retry_logged_2  retry_logging  stand-in  2026-11-13T09:23:25.044646+00:00   
2  retry_logged_3  retry_logging  stand-in  2026-11-13T09:23:26.045589+00:00   
3  retry_logged_4  retry_logging  stand-in  2026-11-13T09:23:32.255904+00:00   
4  retry_logged_5  retry_logging  stand-in  2026-11-13T09:23:33.257244+00:00   

   first_token_ms  total_ms status  retries  tokens_in  tokens_out  \
0             NaN   1001.77     ok        0          4          17   
1             NaN   1000.90     ok        0          4          17   
2             NaN   6210.28     ok        2          4          17   
3             NaN   1001.30     ok        0          4          17   
4             NaN   3179.31     ok        1          4          17   

  retrieval_setting  answer_grade  
0    not_applicable           NaN  
1    not

In [ ]:
import pandas as pd

# Load the original 49 records
final_df = pd.read_csv("results/results.csv")

# Add the 5 recovered retry records
retry_df = pd.read_csv("results/retry_logging.csv")

# Add the two real Groq measurements
groq_rows = pd.DataFrame([
    {
        "request_id": "groq_logged_1",
        "scenario": "real_groq_baseline_logged",
        "source": "groq",
        "start_time": "2026-11-12T15:28:58.165773+00:00",
        "first_token_ms": None,
        "total_ms": 319.41,
        "status": "ok",
        "retries": 0,
        "tokens_in": 80,
        "tokens_out": 70,
        "retrieval_setting": "not_applicable",
        "answer_grade": None
    },
    {
        "request_id": "groq_stream_logged_1",
        "scenario": "real_groq_streaming_logged",
        "source": "groq",
        "start_time": "2026-11-12T15:29:25.551444+00:00",
        "first_token_ms": 320.04,
        "total_ms": 501.18,
        "status": "ok",
        "retries": 0,
        "tokens_in": None,
        "tokens_out": None,
        "retrieval_setting": "not_applicable",
        "answer_grade": None
    }
])

# Avoid duplicates
existing = set(final_df["scenario"].dropna())

retry_df = retry_df[~retry_df["scenario"].isin(existing)]
groq_rows = groq_rows[~groq_rows["scenario"].isin(existing)]

# Combine and save
final_df = pd.concat(
    [final_df, retry_df, groq_rows],
    ignore_index=True
)

final_df["request_id"] = range(1, len(final_df) + 1)

final_df.to_csv("results/results.csv", index=False)

print("TOTAL REQUESTS:", len(final_df))
print("CSV SAVED SUCCESSFULLY!")

TOTAL REQUESTS: 56
CSV SAVED SUCCESSFULLY!


In [ ]:
from google.colab import files
files.download("results/results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Check whether the important project variables still exist

required_items = [
    "documents",
    "chunks",
    "vectorizer",
    "tfidf_matrix",
    "search_papers",
    "ask_research_assistant"
]

for item in required_items:
    if item in globals():
        print(f"✓ {item} is available")
    else:
        print(f"✗ {item} is missing")

✓ documents is available
✓ chunks is available
✓ vectorizer is available
✓ tfidf_matrix is available
✓ search_papers is available
✓ ask_research_assistant is available


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Our three experiment settings
retrieval_configs = [
    {"name": "A", "chunk_size": 900, "top_k": 2, "rerank": False},
    {"name": "B", "chunk_size": 1800, "top_k": 3, "rerank": False},
    {"name": "C", "chunk_size": 3000, "top_k": 5, "rerank": True}
]

def build_retrieval_index(chunk_size):
    new_chunks = []

    for doc in documents:
        text = doc["text"]
        start = 0
        overlap = 300

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end].strip()

            if chunk_text:
                new_chunks.append({
                    "paper": doc["paper"],
                    "page": doc["page"],
                    "text": chunk_text
                })

            start += chunk_size - overlap

    new_vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    new_matrix = new_vectorizer.fit_transform(
        [chunk["text"] for chunk in new_chunks]
    )

    return new_chunks, new_vectorizer, new_matrix


# Build a separate index for each configuration
retrieval_indexes = {}

for config in retrieval_configs:
    new_chunks, new_vectorizer, new_matrix = build_retrieval_index(
        config["chunk_size"]
    )

    retrieval_indexes[config["name"]] = {
        "chunks": new_chunks,
        "vectorizer": new_vectorizer,
        "matrix": new_matrix
    }

    print(
        f"Experiment {config['name']}: "
        f"{len(new_chunks)} chunks created"
    )

Experiment A: 690 chunks created
Experiment B: 312 chunks created
Experiment C: 208 chunks created


In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

evaluation_questions = [
    # Attention Is All You Need
    "Why did the Transformer architecture replace recurrence with attention?",
    "How does scaled dot-product attention calculate attention scores?",
    "What is the purpose of multi-head attention?",
    "How does positional encoding provide information about token positions?",
    "What are the roles of the encoder and decoder in the Transformer?",

    # Moshi
    "What is Moshi and what is its main purpose?",
    "How does Moshi support real-time spoken dialogue?",
    "What is the role of the speech-text representation in Moshi?",
    "How does Moshi handle speech generation and speech understanding?",
    "What makes low-latency dialogue important to the Moshi system?",

    # Dynamic Chunking
    "What problem does dynamic chunking aim to address?",
    "What is hierarchical sequence modeling?",
    "How does dynamic chunking differ from using fixed-size chunks?",
    "Why is chunk selection important in end-to-end sequence modeling?",
    "What benefits does the paper claim for its dynamic chunking approach?"
]

print("Total evaluation questions:", len(evaluation_questions))

Total evaluation questions: 15


In [ ]:
import pandas as pd
import re

def retrieve_with_config(question, config):
    index = retrieval_indexes[config["name"]]

    question_vector = index["vectorizer"].transform([question])
    scores = cosine_similarity(
        question_vector, index["matrix"]
    ).flatten()

    # Retrieve the strongest candidates
    candidate_count = config["top_k"] * 2 if config["rerank"] else config["top_k"]
    best_indices = scores.argsort()[::-1][:candidate_count]

    candidates = []

    for idx in best_indices:
        if scores[idx] <= 0:
            continue

        chunk = index["chunks"][idx].copy()
        chunk["similarity"] = float(scores[idx])
        candidates.append(chunk)

    # Simple lexical reranking for configuration C
    if config["rerank"]:
        question_words = {
            word for word in re.findall(r"\b[a-zA-Z]+\b", question.lower())
            if word not in ENGLISH_STOP_WORDS
        }

        for item in candidates:
            passage_words = set(
                re.findall(r"\b[a-zA-Z]+\b", item["text"].lower())
            )

            overlap = len(question_words & passage_words)
            coverage = overlap / max(len(question_words), 1)

            item["rerank_score"] = (
                0.5 * item["similarity"] + 0.5 * coverage
            )

        candidates.sort(
            key=lambda item: item["rerank_score"],
            reverse=True
        )

    return candidates[:config["top_k"]]


retrieval_results = []

for config in retrieval_configs:
    for question_number, question in enumerate(evaluation_questions, start=1):
        retrieved = retrieve_with_config(question, config)

        for rank, item in enumerate(retrieved, start=1):
            retrieval_results.append({
                "experiment": config["name"],
                "question_number": question_number,
                "question": question,
                "rank": rank,
                "paper": item["paper"],
                "page": item["page"],
                "similarity": round(item["similarity"], 4)
            })

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df.to_csv(
    "results/retrieval_comparison.csv",
    index=False
)

print("Retrieval comparison complete!")
print("Rows saved:", len(retrieval_df))
print(retrieval_df.head(10).to_string(index=False))

Retrieval comparison complete!
Rows saved: 150
experiment  question_number                                                                question  rank                                                          paper  page  similarity
         A                1 Why did the Transformer architecture replace recurrence with attention?     1 Dynamic Chunking for End-to-End Hierarchical Sequence Modeling     9      0.1283
         A                1 Why did the Transformer architecture replace recurrence with attention?     2                                      Attention Is All You Need     2      0.1167
         A                2       How does scaled dot-product attention calculate attention scores?     1                                      Attention Is All You Need     4      0.5427
         A                2       How does scaled dot-product attention calculate attention scores?     2                                      Attention Is All You Need     4      0.3178
         A        

In [ ]:
# Expected paper for each evaluation question
expected_papers = (
    ["Attention Is All You Need"] * 5
    + ["Moshi: a speech-text foundation model for real-time dialogue"] * 5
    + ["Dynamic Chunking for End-to-End Hierarchical Sequence Modeling"] * 5
)

question_targets = pd.DataFrame({
    "question_number": range(1, 16),
    "expected_paper": expected_papers
})

# Add the expected paper to each retrieval result
graded_retrieval = retrieval_df.merge(
    question_targets,
    on="question_number"
)

# Check whether the top result is from the expected paper
graded_retrieval["top1_match"] = (
    (graded_retrieval["rank"] == 1)
    & (graded_retrieval["paper"] == graded_retrieval["expected_paper"])
)

# Check whether at least one of the retrieved passages is from the expected paper
graded_retrieval["expected_paper_found"] = (
    graded_retrieval["paper"] == graded_retrieval["expected_paper"]
)

summary = graded_retrieval.groupby("experiment").agg(
    questions_tested=("question_number", "nunique"),
    top1_correct=("top1_match", "sum"),
    questions_with_expected_paper=("expected_paper_found", "any")
)

summary["top1_accuracy_percent"] = (
    summary["top1_correct"] / summary["questions_tested"] * 100
)

summary["expected_paper_recall_percent"] = (
    summary["questions_with_expected_paper"] / summary["questions_tested"] * 100
)

print(summary.round(2))

            questions_tested  top1_correct  questions_with_expected_paper  \
experiment                                                                  
A                         15            13                           True   
B                         15            15                           True   
C                         15            14                           True   

            top1_accuracy_percent  expected_paper_recall_percent  
experiment                                                        
A                           86.67                           6.67  
B                          100.00                           6.67  
C                           93.33                           6.67  


In [ ]:
# Check whether the expected paper appears anywhere in the retrieved results
recall_per_question = (
    graded_retrieval
    .groupby(["experiment", "question_number"])["expected_paper_found"]
    .any()
    .reset_index()
)

# Calculate correct recall for each experiment
recall_summary = recall_per_question.groupby("experiment").agg(
    questions_tested=("question_number", "count"),
    questions_with_expected_paper=("expected_paper_found", "sum")
)

recall_summary["expected_paper_recall_percent"] = (
    recall_summary["questions_with_expected_paper"]
    / recall_summary["questions_tested"] * 100
)

# Combine with top-1 accuracy
final_retrieval_summary = summary[
    ["top1_correct", "top1_accuracy_percent"]
].join(recall_summary)

print(final_retrieval_summary.round(2))

            top1_correct  top1_accuracy_percent  questions_tested  \
experiment                                                          
A                     13                  86.67                15   
B                     15                 100.00                15   
C                     14                  93.33                15   

            questions_with_expected_paper  expected_paper_recall_percent  
experiment                                                                
A                                      15                          100.0  
B                                      15                          100.0  
C                                      15                          100.0  


In [ ]:
import time

retrieval_performance = []

for config in retrieval_configs:
    for question_number, question in enumerate(
        evaluation_questions, start=1
    ):
        start = time.perf_counter()

        passages = retrieve_with_config(question, config)

        retrieval_time_ms = (
            time.perf_counter() - start
        ) * 1000

        # Estimate context size in words
        context_words = sum(
            len(item["text"].split())
            for item in passages
        )

        retrieval_performance.append({
            "experiment": config["name"],
            "question_number": question_number,
            "retrieval_ms": round(retrieval_time_ms, 2),
            "passages_retrieved": len(passages),
            "context_words": context_words
        })

performance_df = pd.DataFrame(retrieval_performance)

performance_summary = performance_df.groupby("experiment").agg(
    questions=("question_number", "count"),
    retrieval_ms_median=("retrieval_ms", "median"),
    retrieval_ms_p95=("retrieval_ms", lambda x: x.quantile(0.95)),
    average_passages=("passages_retrieved", "mean"),
    average_context_words=("context_words", "mean")
)

performance_df.to_csv(
    "results/retrieval_performance.csv",
    index=False
)

print(performance_summary.round(2))

            questions  retrieval_ms_median  retrieval_ms_p95  \
experiment                                                     
A                  15                 4.25              5.36   
B                  15                 3.59              7.93   
C                  15                 7.11             16.79   

            average_passages  average_context_words  
experiment                                           
A                        2.0                 233.47  
B                        3.0                 771.80  
C                        5.0                1829.00  


In [ ]:
final_retrieval_summary.to_csv(
    "results/retrieval_quality_summary.csv"
)

performance_summary.to_csv(
    "results/retrieval_performance_summary.csv"
)

print("Both summary files saved successfully!")

Both summary files saved successfully!


In [ ]:
# Test one question using Experiment B

test_question = evaluation_questions[0]

config_b = next(
    config for config in retrieval_configs
    if config["name"] == "B"
)

retrieved_b = retrieve_with_config(test_question, config_b)

print("QUESTION:")
print(test_question)

print("\nRETRIEVED PASSAGES:")
for i, item in enumerate(retrieved_b, start=1):
    print(f"\nPassage {i}")
    print("Paper:", item["paper"])
    print("Page:", item["page"])
    print("Similarity:", round(item["similarity"], 4))
    print("Text:", item["text"][:350])

QUESTION:
Why did the Transformer architecture replace recurrence with attention?

RETRIEVED PASSAGES:

Passage 1
Paper: Attention Is All You Need
Page: 2
Similarity: 0.1256
Text: onjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for significantly more parallelization and can reach a new state of the art in
translation quality

Passage 2
Paper: Attention Is All You Need
Page: 4
Similarity: 0.0936
Text: Scaled Dot-Product Attention
Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1
Scaled Dot-Product Atten

Passage 3
Paper: Attention Is All You Ne

In [ ]:
# Prepare evidence from Experiment B

context = ""

for i, item in enumerate(retrieved_b, start=1):
    context += f"""
PASSAGE {i}
Paper: {item['paper']}
Page: {item['page']}
Text: {item['text'][:900]}
"""

prompt = f"""
You are a research assistant.

Answer the question using ONLY the supplied passages.

Rules:
1. Do not introduce outside information.
2. Cite every factual claim using [Paper title, p. page number].
3. If the evidence is insufficient, say so.
4. Do not invent citations.

EVIDENCE:
{context}

QUESTION:
{test_question}
"""

import time

start = time.perf_counter()

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_completion_tokens=300,
    reasoning_effort="low"
)

answer = response.choices[0].message.content

elapsed_ms = (time.perf_counter() - start) * 1000

print("ANSWER:\n")
print(answer)

print("\nAPI LATENCY:", round(elapsed_ms, 2), "ms")
print("INPUT TOKENS:", response.usage.prompt_tokens)
print("OUTPUT TOKENS:", response.usage.completion_tokens)

ANSWER:

The Transformer was designed to eliminate recurrence because an attention‑based mechanism can capture global dependencies between any two positions in the input and output sequences without the need for sequential processing. By relying entirely on attention, the model can compute hidden representations for all positions in parallel, which greatly increases parallelization and enables faster training while still achieving state‑of‑the‑art translation quality【Attention Is All You Need, p. 2】.

API LATENCY: 331.91 ms
INPUT TOKENS: 826
OUTPUT TOKENS: 132


In [ ]:
evaluation_record = {
    "question_number": 1,
    "experiment": "B",
    "question": test_question,
    "answer": answer,
    "answer_grade": "correct",
    "evaluation_note": (
        "Relevant answer and correct page citation. "
        "The phrase 'faster training' is an inference beyond "
        "the retrieved passage's explicit claim of parallelization."
    ),
    "api_latency_ms": 331.91,
    "tokens_in": 826,
    "tokens_out": 132
}

evaluation_df = pd.DataFrame([evaluation_record])

evaluation_df.to_csv(
    "results/answer_quality_evaluation.csv",
    index=False
)

print("First answer evaluation saved!")

First answer evaluation saved!


In [ ]:
import time
import pandas as pd

new_evaluations = []

for question_number in range(2, 6):

    question = evaluation_questions[question_number - 1]

    # Retrieve passages using Experiment B
    retrieved = retrieve_with_config(question, config_b)

    # Prepare evidence
    context = ""

    for i, item in enumerate(retrieved, start=1):
        context += f"""
PASSAGE {i}
Paper: {item['paper']}
Page: {item['page']}
Text: {item['text'][:900]}
"""

    prompt = f"""
You are a research assistant.

Answer using ONLY the supplied passages.

Rules:
1. Do not introduce facts from outside the passages.
2. Cite every factual claim using [Paper title, p. page number].
3. If evidence is insufficient, say so.
4. Never invent citations.

EVIDENCE:
{context}

QUESTION:
{question}
"""

    print("\n" + "=" * 60)
    print(f"QUESTION {question_number}: {question}")

    try:
        start = time.perf_counter()

        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_completion_tokens=300,
            reasoning_effort="low"
        )

        elapsed_ms = (time.perf_counter() - start) * 1000
        generated_answer = response.choices[0].message.content

        print("\nANSWER:")
        print(generated_answer)
        print("\nLatency:", round(elapsed_ms, 2), "ms")
        print("Input tokens:", response.usage.prompt_tokens)
        print("Output tokens:", response.usage.completion_tokens)

        new_evaluations.append({
            "question_number": question_number,
            "experiment": "B",
            "question": question,
            "answer": generated_answer,
            "answer_grade": "",
            "evaluation_note": "",
            "api_latency_ms": round(elapsed_ms, 2),
            "tokens_in": response.usage.prompt_tokens,
            "tokens_out": response.usage.completion_tokens
        })

    except Exception as e:
        print("API ERROR:", str(e))

# Save the new answers
new_df = pd.DataFrame(new_evaluations)

if not new_df.empty:
    new_df.to_csv(
        "results/answers_questions_2_to_5.csv",
        index=False
    )

print("\nSaved", len(new_df), "new answer evaluations.")


QUESTION 2: How does scaled dot-product attention calculate attention scores?

ANSWER:
Scaled dot‑product attention computes the attention scores by taking the dot product of each query vector with all key vectors, scaling the resulting values by the factor \(1/\sqrt{d_k}\), and then applying a softmax function to obtain the weights for the values. This process is described in the paper as: “We compute the dot products of the query with all keys, divide each by \(\sqrt{d_k}\), and apply a softmax function to obtain the weights on the values”【Attention Is All You Need, p. 4】.

Latency: 551.08 ms
Input tokens: 773
Output tokens: 138

QUESTION 3: What is the purpose of multi-head attention?

ANSWER:
The purpose of multi‑head attention is to allow the model to jointly attend to information from different representation subspaces at different positions, thereby avoiding the averaging effect that would occur with a single attention head. This enables the model to capture diverse aspects of 

In [ ]:
import time
import pandas as pd

new_evaluations = []

for question_number in range(2, 6):

    question = evaluation_questions[question_number - 1]

    # Retrieve passages using Experiment B
    retrieved = retrieve_with_config(question, config_b)

    # Prepare evidence
    context = ""

    for i, item in enumerate(retrieved, start=1):
        context += f"""
PASSAGE {i}
Paper: {item['paper']}
Page: {item['page']}
Text: {item['text'][:900]}
"""

    prompt = f"""
You are a research assistant.

Answer using ONLY the supplied passages.

Rules:
1. Do not introduce facts from outside the passages.
2. Cite every factual claim using [Paper title, p. page number].
3. If evidence is insufficient, say so.
4. Never invent citations.

EVIDENCE:
{context}

QUESTION:
{question}
"""

    print("\n" + "=" * 60)
    print(f"QUESTION {question_number}: {question}")

    try:
        start = time.perf_counter()

        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_completion_tokens=300,
            reasoning_effort="low"
        )

        elapsed_ms = (time.perf_counter() - start) * 1000
        generated_answer = response.choices[0].message.content

        print("\nANSWER:")
        print(generated_answer)
        print("\nLatency:", round(elapsed_ms, 2), "ms")
        print("Input tokens:", response.usage.prompt_tokens)
        print("Output tokens:", response.usage.completion_tokens)

        new_evaluations.append({
            "question_number": question_number,
            "experiment": "B",
            "question": question,
            "answer": generated_answer,
            "answer_grade": "",
            "evaluation_note": "",
            "api_latency_ms": round(elapsed_ms, 2),
            "tokens_in": response.usage.prompt_tokens,
            "tokens_out": response.usage.completion_tokens
        })

    except Exception as e:
        print("API ERROR:", str(e))

# Save the new answers
new_df = pd.DataFrame(new_evaluations)

if not new_df.empty:
    new_df.to_csv(
        "results/answers_questions_2_to_5.csv",
        index=False
    )

print("\nSaved", len(new_df), "new answer evaluations.")


QUESTION 2: How does scaled dot-product attention calculate attention scores?

ANSWER:
Scaled dot‑product attention computes the attention scores by taking the dot product of each query vector with all key vectors, scaling the result by the inverse square root of the key dimension \(d_k\), and then applying a softmax to obtain the weights for the values. In formula form, for a set of queries \(Q\), keys \(K\), and values \(V\),

\[
\text{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
\]

[Attention Is All You Need, p. 4]

Latency: 496.09 ms
Input tokens: 773
Output tokens: 149

QUESTION 3: What is the purpose of multi-head attention?

ANSWER:
The purpose of multi‑head attention is to allow the model to jointly attend to information from different representation subspaces at different positions, thereby avoiding the averaging effect that would occur with a single attention head. This enables the model to capture diverse aspects of the input simulta

In [ ]:
# Assign our manual evaluation grades
grades = {
    2: (
        "correct",
        "Accurate explanation and formula; appropriate page 4 citation."
    ),
    3: (
        "correct",
        "Supported by the paper's explanation of multi-head attention on page 5."
    ),
    4: (
        "correct",
        "Accurate overall; 'only way' is slightly stronger than the paper's wording."
    ),
    5: (
        "correct",
        "Accurate encoder and decoder explanation; appropriate page 3 citation."
    )
}

# Add grades to questions 2–5
for index, row in new_df.iterrows():
    q_number = int(row["question_number"])
    new_df.loc[index, "answer_grade"] = grades[q_number][0]
    new_df.loc[index, "evaluation_note"] = grades[q_number][1]

# Load question 1 evaluation
q1_df = pd.read_csv("results/answer_quality_evaluation.csv")

# Combine all five evaluations
all_evaluations = pd.concat(
    [q1_df, new_df],
    ignore_index=True
)

all_evaluations.to_csv(
    "results/answer_quality_evaluation.csv",
    index=False
)

print("Evaluations saved:", len(all_evaluations))
print(all_evaluations[
    ["question_number", "answer_grade", "api_latency_ms"]
])

Evaluations saved: 5
   question_number answer_grade  api_latency_ms
0                1      correct          331.91
1                2      correct          496.09
2                3      correct          180.73
3                4      correct          512.04
4                5      correct          357.59


In [ ]:
import time
import pandas as pd

moshi_evaluations = []

for question_number in range(6, 11):

    question = evaluation_questions[question_number - 1]

    # Retrieve passages using Experiment B
    retrieved = retrieve_with_config(question, config_b)

    # Prepare evidence
    context = ""

    for i, item in enumerate(retrieved, start=1):
        context += f"""
PASSAGE {i}
Paper: {item['paper']}
Page: {item['page']}
Text: {item['text'][:900]}
"""

    prompt = f"""
You are a research assistant.

Answer using ONLY the supplied passages.

Rules:
1. Do not introduce facts from outside the passages.
2. Cite every factual claim using [Paper title, p. page number].
3. If evidence is insufficient, say so.
4. Never invent citations.

EVIDENCE:
{context}

QUESTION:
{question}
"""

    print("\n" + "=" * 60)
    print(f"QUESTION {question_number}: {question}")

    try:
        start = time.perf_counter()

        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_completion_tokens=300,
            reasoning_effort="low"
        )

        elapsed_ms = (time.perf_counter() - start) * 1000
        generated_answer = response.choices[0].message.content

        print("\nANSWER:")
        print(generated_answer)
        print("\nLatency:", round(elapsed_ms, 2), "ms")
        print("Input tokens:", response.usage.prompt_tokens)
        print("Output tokens:", response.usage.completion_tokens)

        moshi_evaluations.append({
            "question_number": question_number,
            "experiment": "B",
            "question": question,
            "answer": generated_answer,
            "answer_grade": "",
            "evaluation_note": "",
            "api_latency_ms": round(elapsed_ms, 2),
            "tokens_in": response.usage.prompt_tokens,
            "tokens_out": response.usage.completion_tokens
        })

    except Exception as e:
        print("API ERROR:", str(e))

# Save these five answers separately
moshi_df = pd.DataFrame(moshi_evaluations)

moshi_df.to_csv(
    "results/answers_questions_6_to_10.csv",
    index=False
)

print("\nSuccessfully generated:", len(moshi_df), "answers")


QUESTION 6: What is Moshi and what is its main purpose?

ANSWER:
Moshi is a speech‑text foundation model designed to enable real‑time spoken dialogue. Its architecture combines a bespoke text language model backbone (Helium), a neural audio codec with residual vector quantization and semantic knowledge distilled from a self‑supervised speech model (Mimi), and a streaming, hierarchical generation of semantic and acoustic tokens for both the user and Moshi, along with time‑aligned text tokens for Moshi when using Inner Monologue [Moshi: a speech-text foundation model for real‑time dialogue, p. 6]. The main purpose of Moshi is to provide a unified, end‑to‑end system that can generate spoken responses in real time, thereby facilitating natural, conversational interactions between a user and the model.

Latency: 320.93 ms
Input tokens: 860
Output tokens: 169

QUESTION 7: How does Moshi support real-time spoken dialogue?

ANSWER:
Moshi supports real‑time spoken dialogue by using a multi‑str

In [ ]:
# Inspect the passages retrieved for Moshi questions 6–10

for question_number in range(6, 11):

    question = evaluation_questions[question_number - 1]

    retrieved = retrieve_with_config(question, config_b)

    print("\n" + "=" * 75)
    print(f"QUESTION {question_number}: {question}")

    for rank, item in enumerate(retrieved, start=1):
        print(f"\nPASSAGE {rank}")
        print("Paper:", item["paper"])
        print("Page:", item["page"])
        print("Similarity:", round(item["similarity"], 4))
        print("Evidence:")
        print(item["text"][:700])


QUESTION 6: What is Moshi and what is its main purpose?

PASSAGE 1
Paper: Moshi: a speech-text foundation model for real-time dialogue
Page: 17
Similarity: 0.0877
Evidence:
Moshi: a speech-text foundation model for real-time dialogue
amounting to a total number of K = 2Q + 1 streams, with Q = 8 in the experiments. A
summary is provided in Figure 4.
Inference of Moshi.
The joint sequence given by Equation 6 is the target for our model-
ing task at train time: At any time step s, the model is input with 0, V1, . . . , Vs−1 and output
an estimated probability distribution ˆVs(0, V1, . . . , Vs−1). At inference time, we sample from
ˆVs,k for all the sub-sequence indexes that corresponds to outputs of Moshi: i.e., for k = 1 for
the text tokens corresponding to Moshi’s speech, and for k ∈{2, . . . , 2+Q} for Moshi’s audio
tokens. In an application setting, prediction

PASSAGE 2
Paper: Moshi: a speech-text foundation model for real-time dialogue
Page: 38
Similarity: 0.0824
Evidence:
peaker c

In [ ]:
moshi_df["answer_grade"] = [
    "correct",
    "correct",
    "wrong",
    "correct",
    "correct"
]

moshi_df["grading_note"] = [
    "Moshi p.6 directly supports the model definition and architecture.",
    "Moshi pp.1 and 6 support full-duplex dialogue and hierarchical streams.",
    "Claims about preserving emotion and non-speech sounds are not fully supported by the retrieved passages shown; citation should also include p.1 for the pipeline comparison.",
    "Moshi p.5 supports Inner Monologue and hierarchical text, semantic and acoustic tokens.",
    "Moshi p.26 supports the latency trade-offs; verify the full passage before retaining the 640 ms detail."
]

moshi_df.to_csv(
    "results/answers_questions_6_to_10.csv",
    index=False
)

print(moshi_df[["question", "answer_grade"]])

                                            question answer_grade
0        What is Moshi and what is its main purpose?      correct
1  How does Moshi support real-time spoken dialogue?      correct
2  What is the role of the speech-text representa...        wrong
3  How does Moshi handle speech generation and sp...      correct
4  What makes low-latency dialogue important to t...      correct


In [ ]:
import time
import pandas as pd

dynamic_chunking_df = []

for question_number in range(11, 16):
    question = evaluation_questions[question_number - 1]

    print(f"\nGenerating answer for Q{question_number}...")
    retrieved = retrieve_with_config(question, config_b)

    if not retrieved:
        print("No passages retrieved. Skipping this question.")
        continue

    context = ""
    for i, item in enumerate(retrieved, start=1):
        context += f"""
PASSAGE {i}
Paper: {item["paper"]}
Page: {item["page"]}
Text: {item["text"][:1000]}
"""

    prompt = f"""
You are a research assistant.
Answer using ONLY the supplied passages.
Do not introduce facts from your general knowledge.
Every factual claim must have a citation in this format:
[Paper title, p. page number]
If the passages do not support an answer, say so.

EVIDENCE:
{context}

QUESTION:
{question}
"""

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_completion_tokens=500,
        reasoning_effort="low"
    )

    total_ms = (time.perf_counter() - start) * 1000
    answer = response.choices[0].message.content

    usage = response.usage

    dynamic_chunking_df.append({
        "question_number": question_number,
        "question": question,
        "answer": answer,
        "total_ms": round(total_ms, 2),
        "tokens_in": usage.prompt_tokens if usage else None,
        "tokens_out": usage.completion_tokens if usage else None,
        "retrieval_setting": "B"
    })

    print(f"Q{question_number} completed — {total_ms:.0f} ms")
    print(answer)

dynamic_chunking_df = pd.DataFrame(dynamic_chunking_df)

dynamic_chunking_df.to_csv(
    "results/answers_questions_11_to_15.csv",
    index=False
)

print("\nSaved answers for Questions 11–15.")


Generating answer for Q11...
Q11 completed — 380 ms
Dynamic chunking is introduced to replace hand‑crafted tokenization heuristics with a learnable, data‑dependent segmentation process.  It aims to compress raw data into shorter, content‑ and context‑aware chunks while preserving semantically important positions, thereby improving efficiency, learnability, and stability of large‑scale sequence models without relying on a fixed tokenizer. [Dynamic Chunking for End-to-End Hierarchical Sequence Modeling, p. 2]

Generating answer for Q12...
Q12 completed — 202 ms
I’m sorry, but none of the supplied passages provide a definition or explanation of “hierarchical sequence modeling.”

Generating answer for Q13...
Q13 completed — 318 ms
Dynamic chunking differs from using fixed‑size chunks in that the boundaries between chunks are **learned and adapted to the data** rather than being predetermined.  
In the H‑Net architecture, a routing module “dynamically draws chunk boundaries” (PASSAGE 1, Fi

In [ ]:
for question_number in range(11, 16):
    question = evaluation_questions[question_number - 1]
    retrieved = retrieve_with_config(question, config_b)

    print("\n" + "=" * 80)
    print(f"QUESTION {question_number}: {question}")

    for i, item in enumerate(retrieved, start=1):
        print(f"\nPASSAGE {i}")
        print("Paper:", item["paper"])
        print("Page:", item["page"])
        print("Evidence:", item["text"][:1000])


QUESTION 11: What problem does dynamic chunking aim to address?

PASSAGE 1
Paper: Dynamic Chunking for End-to-End Hierarchical Sequence Modeling
Page: 3
Evidence: (c) Smoothing Module
n
d
o
f
t
o
k
e
s
T
h
e
_
_
ℰ
Chunking
!
h
e
_
e
n
d
_
o
f
_
t
o
k
e
n
s
!
Dechunkinĝ
.̅
.
'!
'!"#
'!"$
'!"%
1 −'!
1 −'!"#
1 −'!"$
(d) Upsampler
(b) Downsampler
(a) Routing Module
/0
-
/0"
0"#$
*̅
.
+"
Dynamic Chunking
n
_
e
ℳ
Figure 1: (left) Architectural overview of H-Net with a two-stage hierarchical design (𝑆= 2). (right) Dynamic Chunking
(DC). ( bottom-right ) Key components of a chunking layer: (a) a routing module for dynamically drawing chunk
boundaries, and (b) a downsampler that selectively retains vectors based on boundary indicators, reducing sequence length
while preserving semantically significant positions. ( top-right ) Key components of a dechunking layer: (c) a smoothing
module for converting discrete chunks into interpolated representations, and (d) an upsampler that restores compres

In [ ]:
# Add grades to Questions 11–15
dynamic_chunking_df["answer_grade"] = [
    "correct",
    "refused_correctly",
    "wrong",
    "correct",
    "wrong"
]

dynamic_chunking_df["grading_note"] = [
    "Pages 2–3 support the tokenization problem and learned dynamic chunking.",
    "Retrieved passages do not sufficiently explain hierarchical sequence modeling.",
    "Main explanation is plausible, but required paper/page citations are missing.",
    "Page 2 supports the limitations of fixed pooling and benefits of data-dependent chunking.",
    "Broad benefits are supported, but the joint-optimization claim needs a page 33 citation."
]

# Save the final five answers
dynamic_chunking_df.to_csv(
    "results/answers_questions_11_to_15.csv",
    index=False
)

# Load the earlier Questions 1–5 evaluation
first_five = pd.read_csv(
    "results/answer_quality_evaluation.csv"
)

# Combine all three groups
all_evaluations = pd.concat(
    [first_five, moshi_df, dynamic_chunking_df],
    ignore_index=True,
    sort=False
)

# Sort by question number
all_evaluations = all_evaluations.sort_values(
    "question_number"
).reset_index(drop=True)

# Save the combined evaluation
all_evaluations.to_csv(
    "results/answer_quality_evaluation.csv",
    index=False
)

print("Total evaluated questions:", len(all_evaluations))
print("\nGrade breakdown:")
print(all_evaluations["answer_grade"].value_counts())

print("\nSaved combined evaluation successfully.")

Total evaluated questions: 15

Grade breakdown:
answer_grade
correct              11
wrong                 3
refused_correctly     1
Name: count, dtype: int64

Saved combined evaluation successfully.


In [ ]:
print("Evaluation columns:")
print(all_evaluations.columns.tolist())

print("\nFirst 3 evaluation rows:")
display(all_evaluations.head(3))

print("\nRetrieval quality summary:")
display(retrieval_quality_summary)

print("\nRetrieval performance summary:")
display(retrieval_performance_summary)

Evaluation columns:
['question_number', 'experiment', 'question', 'answer', 'answer_grade', 'evaluation_note', 'api_latency_ms', 'tokens_in', 'tokens_out', 'grading_note', 'total_ms', 'retrieval_setting']

First 3 evaluation rows:


,question_number,experiment,question,answer,answer_grade,evaluation_note,api_latency_ms,tokens_in,tokens_out,grading_note,total_ms,retrieval_setting
0,1,B,Why did the Transformer architecture replace r...,The Transformer was designed to eliminate recu...,correct,Relevant answer and correct page citation. The...,331.91,826,132,NaN,NaN,NaN
1,2,B,How does scaled dot-product attention calculat...,Scaled dot‑product attention computes the atte...,correct,Accurate explanation and formula; appropriate ...,496.09,773,149,NaN,NaN,NaN
2,3,B,What is the purpose of multi-head attention?,The purpose of multi‑head attention is to allo...,correct,Supported by the paper's explanation of multi-...,180.73,850,84,NaN,NaN,NaN



Retrieval quality summary:


,experiment,top1_correct,top1_accuracy_percent,questions_tested,questions_with_expected_paper,expected_paper_recall_percent
0,A,13,86.666667,15,15,100.0
1,B,15,100.000000,15,15,100.0
2,C,14,93.333333,15,15,100.0



Retrieval performance summary:


,experiment,questions,retrieval_ms_median,retrieval_ms_p95,average_passages,average_context_words
0,A,15,4.25,5.357,2.0,233.466667
1,B,15,3.59,7.929,3.0,771.800000
2,C,15,7.11,16.791,5.0,1829.000000


In [ ]:
import pandas as pd
import os

quality_path = "results/retrieval_quality_summary.csv"
performance_path = "results/retrieval_performance_summary.csv"

print("Quality file exists:", os.path.exists(quality_path))
print("Performance file exists:", os.path.exists(performance_path))

retrieval_quality_summary = pd.read_csv(quality_path)
retrieval_performance_summary = pd.read_csv(performance_path)

print("\nRETRIEVAL QUALITY")
display(retrieval_quality_summary)

print("\nRETRIEVAL PERFORMANCE")
display(retrieval_performance_summary)

Quality file exists: True
Performance file exists: True

RETRIEVAL QUALITY


,experiment,top1_correct,top1_accuracy_percent,questions_tested,questions_with_expected_paper,expected_paper_recall_percent
0,A,13,86.666667,15,15,100.0
1,B,15,100.000000,15,15,100.0
2,C,14,93.333333,15,15,100.0



RETRIEVAL PERFORMANCE


,experiment,questions,retrieval_ms_median,retrieval_ms_p95,average_passages,average_context_words
0,A,15,4.25,5.357,2.0,233.466667
1,B,15,3.59,7.929,3.0,771.800000
2,C,15,7.11,16.791,5.0,1829.000000


In [ ]:
import os
import pandas as pd

print("Files currently in results folder:")
print(os.listdir("results"))

quality_path = "results/retrieval_quality_summary.csv"
performance_path = "results/retrieval_performance_summary.csv"

if os.path.exists(quality_path):
    retrieval_quality_summary = pd.read_csv(quality_path)
    print("\nRETRIEVAL QUALITY")
    display(retrieval_quality_summary)
else:
    print("\nRetrieval quality summary CSV is missing.")

if os.path.exists(performance_path):
    retrieval_performance_summary = pd.read_csv(performance_path)
    print("\nRETRIEVAL PERFORMANCE")
    display(retrieval_performance_summary)
else:
    print("\nRetrieval performance summary CSV is missing.")

Files currently in results folder:
['burst_traffic.csv', 'failure_recovery.csv', 'answer_quality_evaluation.csv', 'halved_rpm_test.csv', 'queue_test.csv', 'retry_test.csv', 'retrieval_comparison.csv', 'retrieval_performance.csv', 'answers_questions_6_to_10.csv', 'answers_questions_2_to_5.csv', 'daily_budget_test.csv', 'retrieval_performance_summary.csv', 'sustained_overload.csv', 'retrieval_quality_summary.csv', 'results.csv', 'answers_questions_11_to_15.csv', 'retry_logging.csv', 'steady_traffic.csv']

RETRIEVAL QUALITY


,experiment,top1_correct,top1_accuracy_percent,questions_tested,questions_with_expected_paper,expected_paper_recall_percent
0,A,13,86.666667,15,15,100.0
1,B,15,100.000000,15,15,100.0
2,C,14,93.333333,15,15,100.0



RETRIEVAL PERFORMANCE


,experiment,questions,retrieval_ms_median,retrieval_ms_p95,average_passages,average_context_words
0,A,15,4.25,5.357,2.0,233.466667
1,B,15,3.59,7.929,3.0,771.800000
2,C,15,7.11,16.791,5.0,1829.000000


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("results/answer_quality_evaluation.csv")

# Combine the two latency column names
df["measured_latency_ms"] = (
    df["api_latency_ms"].combine_first(df["total_ms"])
)

print("Questions evaluated:", len(df))

print("\nAnswer grades:")
print(df["answer_grade"].value_counts())

print("\nAnswer quality:")
print("Correct:", round((df["answer_grade"] == "correct").mean() * 100, 2), "%")
print("Wrong:", round((df["answer_grade"] == "wrong").mean() * 100, 2), "%")
print(
    "Correctly refused:",
    round((df["answer_grade"] == "refused_correctly").mean() * 100, 2),
    "%"
)

print("\nToken usage:")
print("Average input tokens:", round(df["tokens_in"].mean(), 2))
print("Average output tokens:", round(df["tokens_out"].mean(), 2))

print("\nAnswer generation latency:")
print("P50:", round(df["measured_latency_ms"].quantile(0.50), 2), "ms")
print("P95:", round(df["measured_latency_ms"].quantile(0.95), 2), "ms")
print("P99:", round(df["measured_latency_ms"].quantile(0.99), 2), "ms")

print("\nMissing values:")
print(df[["measured_latency_ms", "tokens_in", "tokens_out"]].isna().sum())

Questions evaluated: 15

Answer grades:
answer_grade
correct              11
wrong                 3
refused_correctly     1
Name: count, dtype: int64

Answer quality:
Correct: 73.33 %
Wrong: 20.0 %
Correctly refused: 6.67 %

Token usage:
Average input tokens: 900.13
Average output tokens: 166.73

Answer generation latency:
P50: 389.6 ms
P95: 670.03 ms
P99: 812.69 ms

Missing values:
measured_latency_ms    0
tokens_in              0
tokens_out             0
dtype: int64


In [ ]:
import pandas as pd

results = pd.read_csv("results/results.csv")

print("Total records:", len(results))

print("\nRecords by scenario:")
print(results["scenario"].value_counts(dropna=False))

print("\nRecords by source:")
print(results["source"].value_counts(dropna=False))

print("\nStatus breakdown:")
print(results["status"].value_counts(dropna=False))

print("\nMissing values:")
print(results.isna().sum())

print("\nMain dataset preview:")
display(results.head(10))

Total records: 56

Records by scenario:
scenario
sustained_overload            12
steady_traffic                 5
queued_concurrency_test        5
burst_traffic                  5
concurrency_retry_test         5
failure_recovery               5
retry_logging                  5
halved_rpm_test                5
daily_budget_test              5
real_groq_streaming            1
real_groq_baseline             1
real_groq_baseline_logged      1
real_groq_streaming_logged     1
Name: count, dtype: int64

Records by source:
source
stand-in    52
groq         4
Name: count, dtype: int64

Status breakdown:
status
ok              52
rate_limited     4
Name: count, dtype: int64

Missing values:
request_id            0
scenario              0
source                0
start_time           42
first_token_ms       54
total_ms              5
status                0
retries              37
tokens_in            49
tokens_out           49
retrieval_setting     0
answer_grade         56
dtype: int64

Main

,request_id,scenario,source,start_time,first_token_ms,total_ms,status,retries,tokens_in,tokens_out,retrieval_setting,answer_grade
0,1,real_groq_baseline,groq,not_recorded,NaN,455.01,ok,0.0,80.0,70.0,not_applicable,NaN
1,2,real_groq_streaming,groq,2026-11-13T09:22:30.194090+00:00,341.99,536.62,ok,0.0,NaN,NaN,not_applicable,NaN
2,3,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147,NaN,NaN,ok,0.0,NaN,NaN,not_applicable,NaN
3,4,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147,NaN,NaN,ok,0.0,NaN,NaN,not_applicable,NaN
4,5,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147,NaN,NaN,ok,2.0,NaN,NaN,not_applicable,NaN
5,6,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147,NaN,NaN,ok,1.0,NaN,NaN,not_applicable,NaN
6,7,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147,NaN,NaN,ok,3.0,NaN,NaN,not_applicable,NaN
7,8,queued_concurrency_test,stand-in,NaN,NaN,1001.25,ok,NaN,NaN,NaN,not_applicable,NaN
8,9,queued_concurrency_test,stand-in,NaN,NaN,2503.13,ok,NaN,NaN,NaN,not_applicable,NaN
9,10,queued_concurrency_test,stand-in,NaN,NaN,3502.08,ok,NaN,NaN,NaN,not_applicable,NaN


In [ ]:
audit = results.groupby("scenario").agg(
    records=("request_id", "count"),
    successes=("status", lambda x: (x == "ok").sum()),
    rate_limited=("status", lambda x: (x == "rate_limited").sum()),
    missing_total_ms=("total_ms", lambda x: x.isna().sum()),
    missing_start_time=("start_time", lambda x: x.isna().sum()),
    missing_retries=("retries", lambda x: x.isna().sum()),
    missing_tokens_in=("tokens_in", lambda x: x.isna().sum()),
    missing_tokens_out=("tokens_out", lambda x: x.isna().sum())
).reset_index()

display(audit)

,scenario,records,successes,rate_limited,missing_total_ms,missing_start_time,missing_retries,missing_tokens_in,missing_tokens_out
0,burst_traffic,5,5,0,0,5,5,5,5
1,concurrency_retry_test,5,5,0,5,0,0,5,5
2,daily_budget_test,5,3,2,0,5,5,5,5
3,failure_recovery,5,5,0,0,5,0,5,5
4,halved_rpm_test,5,3,2,0,5,5,5,5
5,queued_concurrency_test,5,5,0,0,5,5,5,5
6,real_groq_baseline,1,1,0,0,0,0,0,0
7,real_groq_baseline_logged,1,1,0,0,0,0,0,0
8,real_groq_streaming,1,1,0,0,0,0,1,1
9,real_groq_streaming_logged,1,1,0,0,0,0,1,1


In [ ]:
import pandas as pd

results = pd.read_csv("results/results.csv")

# Remove duplicate copies of the two Groq measurements
clean = results[
    ~results["scenario"].isin([
        "real_groq_baseline_logged",
        "real_groq_streaming_logged"
    ])
].copy()

latency_scenarios = [
    "steady_traffic",
    "burst_traffic",
    "sustained_overload",
    "queued_concurrency_test",
    "retry_logging",
    "real_groq_baseline",
    "real_groq_streaming"
]

latency_rows = []

for scenario in latency_scenarios:
    subset = clean[
        (clean["scenario"] == scenario) &
        (clean["total_ms"].notna())
    ]

    if len(subset) == 0:
        continue

    latency_rows.append({
        "scenario": scenario,
        "measured_requests": len(subset),
        "p50_ms": round(subset["total_ms"].quantile(0.50), 2),
        "p95_ms": round(subset["total_ms"].quantile(0.95), 2),
        "p99_ms": round(subset["total_ms"].quantile(0.99), 2)
    })

latency_summary = pd.DataFrame(latency_rows)
display(latency_summary)

,scenario,measured_requests,p50_ms,p95_ms,p99_ms
0,steady_traffic,5,1001.36,1001.53,1001.55
1,burst_traffic,5,3503.62,5805.47,5965.67
2,sustained_overload,12,8006.82,14187.40,14847.76
3,queued_concurrency_test,5,3502.08,5804.38,5964.26
4,retry_logging,5,1001.77,5604.09,6089.04
5,real_groq_baseline,1,455.01,455.01,455.01
6,real_groq_streaming,1,536.62,536.62,536.62


In [ ]:
outcomes = clean.groupby("scenario").agg(
    requests=("request_id", "count"),
    succeeded=("status", lambda x: (x == "ok").sum()),
    rate_limited=("status", lambda x: (x == "rate_limited").sum()),
    retries_recorded=("retries", "sum"),
    rows_with_retry_data=("retries", "count")
).reset_index()

outcomes["success_percent"] = (
    outcomes["succeeded"] / outcomes["requests"] * 100
).round(2)

outcomes["rate_limited_percent"] = (
    outcomes["rate_limited"] / outcomes["requests"] * 100
).round(2)

display(outcomes)

,scenario,requests,succeeded,rate_limited,retries_recorded,rows_with_retry_data,success_percent,rate_limited_percent
0,burst_traffic,5,5,0,0.0,0,100.0,0.0
1,concurrency_retry_test,5,5,0,6.0,5,100.0,0.0
2,daily_budget_test,5,3,2,0.0,0,60.0,40.0
3,failure_recovery,5,5,0,3.0,5,100.0,0.0
4,halved_rpm_test,5,3,2,0.0,0,60.0,40.0
5,queued_concurrency_test,5,5,0,0.0,0,100.0,0.0
6,real_groq_baseline,1,1,0,0.0,1,100.0,0.0
7,real_groq_streaming,1,1,0,0.0,1,100.0,0.0
8,retry_logging,5,5,0,3.0,5,100.0,0.0
9,steady_traffic,5,5,0,0.0,0,100.0,0.0


In [ ]:
clean.to_csv("results/results_clean.csv", index=False)

print("Clean results saved.")
print("Rows:", len(clean))
print("File: results/results_clean.csv")

Clean results saved.
Rows: 54
File: results/results_clean.csv


In [ ]:
import pandas as pd
import os

files_to_check = [
    "results/retry_test.csv",
    "results/retry_logging.csv",
    "results/failure_recovery.csv",
    "results/queue_test.csv"
]

for file in files_to_check:
    print("\n" + "=" * 50)
    print("FILE:", file)

    if os.path.exists(file):
        df = pd.read_csv(file)
        print("Rows:", len(df))
        display(df)
    else:
        print("File not found")


FILE: results/retry_test.csv
Rows: 5


,request_id,retries,status,scenario,source,start_time
0,1,0,ok,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147
1,2,0,ok,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147
2,3,2,ok,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147
3,4,1,ok,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147
4,5,3,ok,concurrency_retry_test,stand-in,2026-11-13T09:22:29.592147



FILE: results/retry_logging.csv
Rows: 5


,request_id,scenario,source,start_time,first_token_ms,total_ms,status,retries,tokens_in,tokens_out,retrieval_setting,answer_grade
0,retry_logged_1,retry_logging,stand-in,2026-11-13T09:23:24.042821+00:00,NaN,1001.77,ok,0,4,17,not_applicable,NaN
1,retry_logged_2,retry_logging,stand-in,2026-11-13T09:23:25.044646+00:00,NaN,1000.90,ok,0,4,17,not_applicable,NaN
2,retry_logged_3,retry_logging,stand-in,2026-11-13T09:23:26.045589+00:00,NaN,6210.28,ok,2,4,17,not_applicable,NaN
3,retry_logged_4,retry_logging,stand-in,2026-11-13T09:23:32.255904+00:00,NaN,1001.30,ok,0,4,17,not_applicable,NaN
4,retry_logged_5,retry_logging,stand-in,2026-11-13T09:23:33.257244+00:00,NaN,3179.31,ok,1,4,17,not_applicable,NaN



FILE: results/failure_recovery.csv
Rows: 5


,request_id,total_ms,status,retries
0,1,3061.93,ok,1
1,2,1001.36,ok,0
2,3,1001.27,ok,0
3,4,1001.31,ok,0
4,5,6376.50,ok,2



FILE: results/queue_test.csv
Rows: 5


,request_id,queue_wait_ms,total_ms,status,scenario,source,peak_queue
0,1,0.03,1001.25,ok,queued_concurrency_test,stand-in,3
1,2,0.01,2503.13,ok,queued_concurrency_test,stand-in,3
2,3,1001.15,3502.08,ok,queued_concurrency_test,stand-in,3
3,4,2503.11,5004.98,ok,queued_concurrency_test,stand-in,3
4,5,3502.08,6004.23,ok,queued_concurrency_test,stand-in,3


In [ ]:
import pandas as pd

df = pd.read_csv("results/retrieval_comparison.csv")

print("Total retrieval records:", len(df))
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nFirst 15 records:")
display(df.head(15))

Total retrieval records: 150

Available columns:
['experiment', 'question_number', 'question', 'rank', 'paper', 'page', 'similarity']

First 15 records:


,experiment,question_number,question,rank,paper,page,similarity
0,A,1,Why did the Transformer architecture replace r...,1,Dynamic Chunking for End-to-End Hierarchical S...,9,0.1283
1,A,1,Why did the Transformer architecture replace r...,2,Attention Is All You Need,2,0.1167
2,A,2,How does scaled dot-product attention calculat...,1,Attention Is All You Need,4,0.5427
3,A,2,How does scaled dot-product attention calculat...,2,Attention Is All You Need,4,0.3178
4,A,3,What is the purpose of multi-head attention?,1,Attention Is All You Need,5,0.3482
5,A,3,What is the purpose of multi-head attention?,2,Attention Is All You Need,5,0.2507
6,A,4,How does positional encoding provide informati...,1,Attention Is All You Need,6,0.1344
7,A,4,How does positional encoding provide informati...,2,Attention Is All You Need,6,0.1180
8,A,5,What are the roles of the encoder and decoder ...,1,Dynamic Chunking for End-to-End Hierarchical S...,19,0.2397
9,A,5,What are the roles of the encoder and decoder ...,2,Attention Is All You Need,5,0.2006


In [ ]:
expected_papers = {
    **{i: "Attention Is All You Need" for i in range(1, 6)},
    **{i: "Moshi: a speech-text foundation model for real-time dialogue" for i in range(6, 11)},
    **{i: "Dynamic Chunking for End-to-End Hierarchical Sequence Modeling" for i in range(11, 16)}
}

top_results = df[df["rank"] == 1].copy()

top_results["expected_paper"] = top_results["question_number"].map(expected_papers)
top_results["correct_paper"] = (
    top_results["paper"] == top_results["expected_paper"]
)

display(
    top_results[
        ["experiment", "question_number", "question",
         "paper", "page", "similarity", "correct_paper"]
    ].sort_values(["question_number", "experiment"])
)

,experiment,question_number,question,paper,page,similarity,correct_paper
0,A,1,Why did the Transformer architecture replace r...,Dynamic Chunking for End-to-End Hierarchical S...,9,0.1283,False
30,B,1,Why did the Transformer architecture replace r...,Attention Is All You Need,2,0.1256,True
75,C,1,Why did the Transformer architecture replace r...,Attention Is All You Need,2,0.1045,True
2,A,2,How does scaled dot-product attention calculat...,Attention Is All You Need,4,0.5427,True
33,B,2,How does scaled dot-product attention calculat...,Attention Is All You Need,4,0.5552,True
80,C,2,How does scaled dot-product attention calculat...,Attention Is All You Need,4,0.5030,True
4,A,3,What is the purpose of multi-head attention?,Attention Is All You Need,5,0.3482,True
36,B,3,What is the purpose of multi-head attention?,Attention Is All You Need,5,0.2686,True
85,C,3,What is the purpose of multi-head attention?,Attention Is All You Need,5,0.2174,True
6,A,4,How does positional encoding provide informati...,Attention Is All You Need,6,0.1344,True


In [ ]:
import pandas as pd

quality = pd.read_csv("results/retrieval_quality_summary.csv")
performance = pd.read_csv("results/retrieval_performance_summary.csv")
answers = pd.read_csv("results/answer_quality_evaluation.csv")

print("RETRIEVAL QUALITY — PAPER MATCHING")
display(quality)

print("\nRETRIEVAL PERFORMANCE — SPEED AND CONTEXT")
display(performance)

print("\nANSWER QUALITY — 15 QUESTIONS")
print(answers["answer_grade"].value_counts())

print("\nAVERAGE TOKEN USAGE")
print("Input:", round(answers["tokens_in"].mean(), 2))
print("Output:", round(answers["tokens_out"].mean(), 2))

print("\nANSWER GENERATION LATENCY")
print("P50:", round(answers["total_ms"].quantile(0.50), 2), "ms")
print("P95:", round(answers["total_ms"].quantile(0.95), 2), "ms")
print("P99:", round(answers["total_ms"].quantile(0.99), 2), "ms")

RETRIEVAL QUALITY — PAPER MATCHING


,experiment,top1_correct,top1_accuracy_percent,questions_tested,questions_with_expected_paper,expected_paper_recall_percent
0,A,13,86.666667,15,15,100.0
1,B,15,100.000000,15,15,100.0
2,C,14,93.333333,15,15,100.0



RETRIEVAL PERFORMANCE — SPEED AND CONTEXT


,experiment,questions,retrieval_ms_median,retrieval_ms_p95,average_passages,average_context_words
0,A,15,4.25,5.357,2.0,233.466667
1,B,15,3.59,7.929,3.0,771.800000
2,C,15,7.11,16.791,5.0,1829.000000



ANSWER QUALITY — 15 QUESTIONS
answer_grade
correct              11
wrong                 3
refused_correctly     1
Name: count, dtype: int64

AVERAGE TOKEN USAGE
Input: 900.13
Output: 166.73

ANSWER GENERATION LATENCY
P50: 380.31 ms
P95: 446.89 ms
P99: 458.35 ms


In [ ]:
import pandas as pd

df = pd.read_csv("results/results_clean.csv")

print("Rows:", len(df))
print("\nStatus counts:")
print(df["status"].value_counts(dropna=False))

print("\nScenario counts:")
print(df["scenario"].value_counts())

print("\nSource counts:")
print(df["source"].value_counts())

print("\nMissing values:")
print(df.isna().sum())

Rows: 54

Status counts:
status
ok              50
rate_limited     4
Name: count, dtype: int64

Scenario counts:
scenario
sustained_overload         12
queued_concurrency_test     5
concurrency_retry_test      5
halved_rpm_test             5
daily_budget_test           5
steady_traffic              5
burst_traffic               5
failure_recovery            5
retry_logging               5
real_groq_streaming         1
real_groq_baseline          1
Name: count, dtype: int64

Source counts:
source
stand-in    52
groq         2
Name: count, dtype: int64

Missing values:
request_id            0
scenario              0
source                0
start_time           42
first_token_ms       53
total_ms              5
status                0
retries              37
tokens_in            48
tokens_out           48
retrieval_setting     0
answer_grade         54
dtype: int64


In [ ]:
import pandas as pd

df = pd.read_csv("results/results_clean.csv")

summary = df.groupby(["scenario", "source"]).agg(
    requests=("request_id", "count"),
    successes=("status", lambda x: (x == "ok").sum()),
    rate_limited=("status", lambda x: (x == "rate_limited").sum()),
    timeouts=("status", lambda x: (x == "timeout").sum()),
    errors=("status", lambda x: (x == "error").sum()),
    average_total_ms=("total_ms", "mean"),
    p50_total_ms=("total_ms", "median"),
    p95_total_ms=("total_ms", lambda x: x.quantile(0.95)),
    retries_recorded=("retries", "sum"),
    rows_with_retry_data=("retries", "count"),
    peak_queue=("request_id", "count")
).reset_index()

display(summary.round(2))

,scenario,source,requests,successes,rate_limited,timeouts,errors,average_total_ms,p50_total_ms,p95_total_ms,retries_recorded,rows_with_retry_data,peak_queue
0,burst_traffic,stand-in,5,5,0,0,0,3603.63,3503.62,5805.47,0.0,0,5
1,concurrency_retry_test,stand-in,5,5,0,0,0,NaN,NaN,NaN,6.0,5,5
2,daily_budget_test,stand-in,5,3,2,0,0,600.94,1001.01,1002.21,0.0,0,5
3,failure_recovery,stand-in,5,5,0,0,0,2488.47,1001.36,5713.59,3.0,5,5
4,halved_rpm_test,stand-in,5,3,2,0,0,600.93,1001.29,1001.65,0.0,0,5
5,queued_concurrency_test,stand-in,5,5,0,0,0,3603.13,3502.08,5804.38,0.0,0,5
6,real_groq_baseline,groq,1,1,0,0,0,455.01,455.01,455.01,0.0,1,1
7,real_groq_streaming,groq,1,1,0,0,0,536.62,536.62,536.62,0.0,1,1
8,retry_logging,stand-in,5,5,0,0,0,2478.71,1001.77,5604.09,3.0,5,5
9,steady_traffic,stand-in,5,5,0,0,0,1001.38,1001.36,1001.53,0.0,0,5


In [ ]:
import os
import zipfile
from google.colab import files

# Folder containing your experiment CSV files
results_folder = "/content/results"

# Create a ZIP containing every CSV in the results folder
zip_path = "/content/relay_all_results.zip"

csv_files = [
    os.path.join(results_folder, f)
    for f in os.listdir(results_folder)
    if f.lower().endswith(".csv")
]

if not csv_files:
    print("No CSV files found in:", results_folder)
else:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file_path in csv_files:
            zipf.write(file_path, arcname=os.path.basename(file_path))

    print(f"Collected {len(csv_files)} CSV files:")
    for file_path in csv_files:
        print("-", os.path.basename(file_path))

    files.download(zip_path)

Collected 19 CSV files:
- burst_traffic.csv
- failure_recovery.csv
- answer_quality_evaluation.csv
- halved_rpm_test.csv
- queue_test.csv
- retry_test.csv
- retrieval_comparison.csv
- retrieval_performance.csv
- answers_questions_6_to_10.csv
- answers_questions_2_to_5.csv
- daily_budget_test.csv
- retrieval_performance_summary.csv
- sustained_overload.csv
- results_clean.csv
- retrieval_quality_summary.csv
- results.csv
- answers_questions_11_to_15.csv
- retry_logging.csv
- steady_traffic.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>